In [ ]:
%load_ext autoreload
%autoreload 2

### Perform dataset leakage analysis and organize all data used across all benchmarks and updating experiments

 - Get the train/test datasets from each model we benchmarked and identify train/test overlaps in our benchmark sets.
 - Get the model versions used and format in a proper table.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name not in ("CLSBN26", "PepCL_Manuscript"):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"
FIGURES_DIR = PROJECT_ROOT / "Figures"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data
FIGURES_DIR: /home/pchati/PepCL_Manuscript/Figures
PROJECT_ROOT in sys.path: True


In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.plotting import *
from src.pepcl_logging import *

# misc
import warnings
warnings.filterwarnings("ignore")

### Figure output directory

In [ ]:
# create figures and data dir
data_leakage_figures_dir=f"{FIGURES_DIR}/Revisions/Data_Leakage_Figures"
data_leakage_figures_data_dir=f"{data_leakage_figures_dir}/Associated_Data"

os.makedirs(data_leakage_figures_dir, exist_ok=True)
os.makedirs(data_leakage_figures_data_dir, exist_ok=True)

### Model version table

 - Need a table with model versions used.

In [ ]:
def plot_df_table(
    df,
    columns=None,
    column_labels=None,
    col_widths=None,

    # Figure / fonts.
    fig_size=None,
    dpi=300,
    title="",
    title_fontsize=16,
    header_fontsize=11,
    body_fontsize=11,
    font_family="Nimbus Sans",

    # Table geometry.
    table_bbox=(0.0,0.02,1.0,0.92),
    row_height_scale=1.45,
    cell_pad=None,

    # First-column grouping.
    merge_repeated_first_col=False,
    first_col_text_pad=0.03,

    # Bold styling.
    bold_columns=None,
    bold_rows=None,
    bold_cells=None,

    # Horizontal rules.
    top_linewidth=1.5,
    header_linewidth=0.8,
    bottom_linewidth=1.5,
    row_linewidth=0.5,
    row_lines=None,
    row_line_alpha=.35,
    line_color="black",
    independent_lines=True,

    # Output.
    save_path=None,
    show_plot=True,
):
    """
    Plot a dataframe using manuscript-style table formatting.

    If merge_repeated_first_col=True, consecutive identical values in the
    first displayed column are shown once, vertically centered across their
    contiguous row block and left-aligned. All other columns are center-aligned.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataframe to display.

    columns : list, optional
        Columns to include and their order. Defaults to all dataframe columns.

    column_labels : dict or list, optional
        Display labels. A dict maps original dataframe columns to labels.

    col_widths : list, optional
        Relative widths of displayed columns.

    merge_repeated_first_col : bool
        Merge consecutive identical values in the first displayed column.

    first_col_text_pad : float
        Fraction of the first-column width used as left padding for merged
        first-column labels.

    bold_columns : list, optional
        Entire body columns to render bold.

    bold_rows : list, optional
        Zero-based displayed dataframe row positions to render bold.

    bold_cells : list, optional
        Individual cells as (row_position, column) tuples.

    row_lines : list, optional
        Zero-based displayed row positions AFTER which a horizontal separator
        is drawn.

    Returns
    -------
    fig, ax, tbl, display_df
    """
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.font_manager import FontProperties
    from matplotlib.lines import Line2D

    if not isinstance(df,pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")

    display=df.copy()

    if columns is None:
        columns=list(display.columns)
    else:
        missing=[c for c in columns if c not in display.columns]
        if missing:
            raise ValueError(f"columns contains missing dataframe columns: {missing}")
        display=display.loc[:,columns].copy()

    columns=list(display.columns)
    ncols=len(columns)

    if ncols==0:
        raise ValueError("No columns available to plot.")

    if column_labels is None:
        final_labels=[str(c) for c in columns]
    elif isinstance(column_labels,dict):
        final_labels=[column_labels.get(c,str(c)) for c in columns]
    else:
        if len(column_labels)!=ncols:
            raise ValueError(f"column_labels must have length {ncols}, got {len(column_labels)}.")
        final_labels=list(column_labels)

    if col_widths is not None and len(col_widths)!=ncols:
        raise ValueError(f"col_widths must have length {ncols}, got {len(col_widths)}.")

    if fig_size is None:
        fig_size=(max(8,2.0*ncols),max(2.5,.42*(len(display)+2)))

    bold_columns=[] if bold_columns is None else list(bold_columns)
    bold_rows=[] if bold_rows is None else list(bold_rows)
    bold_cells=[] if bold_cells is None else list(bold_cells)

    missing=[c for c in bold_columns if c not in columns]
    if missing:
        raise ValueError(f"bold_columns contains missing displayed columns: {missing}")

    missing=[c for _,c in bold_cells if c not in columns]
    if missing:
        raise ValueError(f"bold_cells contains missing displayed columns: {missing}")

    for row in bold_rows:
        if row<0 or row>=len(display):
            raise ValueError(f"bold_rows contains invalid row position {row}.")

    for row,column in bold_cells:
        if row<0 or row>=len(display):
            raise ValueError(f"bold_cells contains invalid row position {row}.")
        if column not in columns:
            raise ValueError(f"bold_cells contains missing displayed column: {column}")

    matplotlib.rcParams["pdf.fonttype"]=42
    matplotlib.rcParams["ps.fonttype"]=42
    matplotlib.rcParams["svg.fonttype"]="none"
    matplotlib.rcParams["font.family"]="sans-serif"
    matplotlib.rcParams["font.sans-serif"]=[
        font_family,
        "Nimbus Sans",
        "Nimbus Sans L",
        "DejaVu Sans",
    ]

    fp_body=FontProperties(family=font_family,size=body_fontsize)
    fp_body_bold=FontProperties(family=font_family,size=body_fontsize,weight="bold")
    fp_header=FontProperties(family=font_family,size=header_fontsize,weight="bold")
    fp_title=FontProperties(family=font_family,size=title_fontsize)

    fig,ax=plt.subplots(figsize=fig_size,dpi=dpi)
    ax.axis("off")

    if title is not None:
        ax.text(
            0.0,1.02,title,
            transform=ax.transAxes,
            ha="left",va="bottom",
            fontproperties=fp_title
        )

    # If grouping the first column, remove its normal cell text and redraw
    # each contiguous block separately after the table geometry is known.
    table_display=display.copy()
    first_col=columns[0]

    if merge_repeated_first_col:
        table_display.iloc[:,0]=""

    tbl=ax.table(
        cellText=table_display.values.tolist(),
        colLabels=final_labels,
        cellLoc="center",
        colLoc="center",
        colWidths=col_widths,
        bbox=table_bbox,
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(body_fontsize)
    tbl.scale(1.0,row_height_scale)

    nrows=len(display)+1

    # Remove matplotlib cell borders/backgrounds.
    for (r,c),cell in tbl.get_celld().items():
        cell.visible_edges=""
        cell.set_linewidth(0)
        cell.set_facecolor("white")
        cell.get_text().set_fontproperties(fp_body)
        if cell_pad is not None:
            cell.PAD=cell_pad

    # Headers: bold and centered.
    for c in range(ncols):
        txt=tbl[(0,c)].get_text()
        txt.set_fontproperties(fp_header)
        txt.set_ha("center")

    # Body: first column left-aligned, all other columns centered.
    for r in range(1,nrows):
        tbl[(r,0)].get_text().set_ha("left")
        for c in range(1,ncols):
            tbl[(r,c)].get_text().set_ha("center")

    # Bold complete columns.
    for column in bold_columns:
        c=columns.index(column)
        if merge_repeated_first_col and c==0:
            continue
        for r in range(1,nrows):
            tbl[(r,c)].get_text().set_fontproperties(fp_body_bold)

    # Bold complete rows.
    for row in bold_rows:
        table_row=row+1
        for c in range(ncols):
            if merge_repeated_first_col and c==0:
                continue
            tbl[(table_row,c)].get_text().set_fontproperties(fp_body_bold)

    # Bold individual cells.
    for row,column in bold_cells:
        c=columns.index(column)
        if merge_repeated_first_col and c==0:
            continue
        tbl[(row+1,c)].get_text().set_fontproperties(fp_body_bold)

    fig.canvas.draw()

    # ==============================================================
    # Merged consecutive first-column labels
    # ==============================================================
    if merge_repeated_first_col and len(display)>0:
        values=display.iloc[:,0].tolist()
        groups=[]
        start=0

        for i in range(1,len(values)+1):
            if i==len(values) or values[i]!=values[start]:
                groups.append((start,i-1,values[start]))
                start=i

        for start,end,value in groups:
            r1=start+1
            r2=end+1

            cell_top=tbl[(r1,0)]
            cell_bottom=tbl[(r2,0)]

            x=cell_top.get_x()
            width=cell_top.get_width()
            top=cell_top.get_y()+cell_top.get_height()
            bottom=cell_bottom.get_y()
            y=(top+bottom)/2

            # Determine whether the merged label should be bold.
            is_bold=(
                first_col in bold_columns
                or any(r in bold_rows for r in range(start,end+1))
                or any(
                    row>=start and row<=end and column==first_col
                    for row,column in bold_cells
                )
            )

            ax.text(
                x+first_col_text_pad*width,
                y,
                "" if pd.isna(value) else str(value),
                transform=ax.transAxes,
                ha="left",
                va="center",
                fontproperties=fp_body_bold if is_bold else fp_body,
                zorder=10,
            )

    # ==============================================================
    # Independent table rules
    # ==============================================================
    left=tbl[(0,0)].get_x()
    right=tbl[(0,ncols-1)].get_x()+tbl[(0,ncols-1)].get_width()
    header_top=tbl[(0,0)].get_y()+tbl[(0,0)].get_height()
    header_bottom=tbl[(0,0)].get_y()
    bottom=tbl[(nrows-1,0)].get_y()

    def _add_rule(y,linewidth,alpha=1.0):
        if independent_lines:
            line=Line2D(
                [left,right],[y,y],
                transform=ax.transAxes,
                color=line_color,
                linewidth=linewidth,
                alpha=alpha,
                linestyle="-",
                solid_capstyle="butt",
                clip_on=False,
                zorder=5,
            )
            ax.add_line(line)
            return line

        return ax.hlines(
            y,left,right,
            transform=ax.transAxes,
            linewidth=linewidth,
            color=line_color,
            alpha=alpha,
            zorder=5
        )

    _add_rule(header_top,top_linewidth)
    _add_rule(header_bottom,header_linewidth)

    if row_lines is not None:
        for row in row_lines:
            if row<0 or row>=len(display)-1:
                raise ValueError(
                    f"row_lines contains invalid row position {row}; "
                    "a separator must occur before the final row."
                )
            y=tbl[(row+1,0)].get_y()
            _add_rule(y,row_linewidth,alpha=row_line_alpha)

    _add_rule(bottom,bottom_linewidth)

    if save_path is not None:
        fig.savefig(f"{save_path}.pdf",dpi=dpi,bbox_inches="tight")
        fig.savefig(f"{save_path}.png",dpi=dpi,bbox_inches="tight")

    if show_plot:
        plt.show()

    return fig,ax,tbl,display

In [ ]:
# get table
model_version_table=pd.read_csv(f"{DATA_DIR}/External_Model_Versions_Table.csv")

In [ ]:
# # plot model versions table
# fig,ax,tbl,display_df=plot_df_table(
#     model_version_table,
#     columns=[
#         "Model",
#         "Version",
#         "Release date",
#         "Settings",
#         "Train source",
#     ],
#     col_widths=[0.07,0.10,0.10,0.14,0.14],
#     merge_repeated_first_col=True,
#     first_col_text_pad=0.03,
#     bold_columns=["Model"],
#     title="",
#     fig_size=(18,7),
#     dpi=300,
#     header_fontsize=11,
#     body_fontsize=11,
#     title_fontsize=16,
#     table_bbox=(0.0,0.02,1.0,0.92),
#     row_height_scale=1.45,
#     top_linewidth=1.5,
#     header_linewidth=0.8,
#     bottom_linewidth=1.5,
#     row_lines=[0,1,2,3,4,7,8,10],
#     row_linewidth=0.5,
#     row_line_alpha=0.35,
#     independent_lines=True,
#     # save_path=f"{data_leakage_figures_dir}/model_versions_table",
# )


### Load data

#### MHC pseudosequences

In [ ]:
# get mhc pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### All MHCPrime train/validation/test data

 - Used for creating table with counts and specifications per experiment

In [ ]:
# import ms train/val
ms_train_data=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/ms_train_data_remaining_minimal.csv", index_col=0)
ms_val_data=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/ms_val_data.csv", index_col=0)

In [ ]:
# process
ms_train_data.seq=ms_train_data.seq.str.replace("$", "")
ms_val_data.seq=ms_val_data.seq.str.replace("$", "")

# merge
ms_train_val_data=pd.concat([ms_train_data, ms_val_data])

# create seq allele col
ms_train_val_data=create_seq_allele_column(ms_train_val_data)

In [ ]:
"""
DFs:

 - ms_train_val_data
"""

'\nDFs:\n\n - ms_train_val_data\n'

#### MS and assay-specific model benchmarking test datasets

 - Primarily categorize overlap between the test datasets used for main model benchmarking and any other datasets run on the external model.

In [ ]:
# ms test data
ms_test_data=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/ms_test_data.csv", index_col=0)

In [ ]:
# yd test set
holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
holec_test_set = pd.read_csv(f"{holec_datasets_dir}/holec_test_set.csv", index_col=0)

In [ ]:
# episcan r9/rt and cov
episcan_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
episcan_r9_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv", index_col=0)
episcan_rt_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv", index_col=0)
sd3_formatted=pd.read_csv(f"{episcan_train_test_dir}/sd3_formatted.csv", index_col=0)

In [ ]:
# ba test sets

# cls
ba_full_cls_test_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_cls_test_set.csv", index_col=0)

# con
ba_full_con_test_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_con_test_set.csv", index_col=0)

In [ ]:
# escape-seq test sets

# cls
escape_seq_full_cls_test_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_seq_full_cls_test_set.csv", index_col=0)

# con
escape_seq_full_con_test_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_seq_full_con_test_set.csv", index_col=0)

# reported neoantigens (for NMP)
escapeseq_reported_neoantigens_expanded=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escapeseq_reported_neoantigens_expanded_opt.csv", index_col=0)

In [ ]:
%%time
# standardize format to seq, allele, label (and ba_value or escore for continuous data)
test_datasets_for_overlap_analysis={
    "ms": ms_test_data,
    "yd": holec_test_set,
    "episcan_r9": episcan_r9_test_set,
    "episcan_rt": episcan_rt_test_set,
    "episcan_cov": sd3_formatted,
    "ba_cls": ba_full_cls_test_set,
    "ba_con": ba_full_con_test_set,
    "escape_cls": escape_seq_full_cls_test_set,
    "escape_con": escape_seq_full_con_test_set,
    "escape_neo": escapeseq_reported_neoantigens_expanded,
}

def filter_to_cols(df, cols):
    keep = [c for c in cols if c in df.columns]
    return df[keep].copy()

target_cols=["seq", "allele", "label", "ba_value", "escore"]

for df_name, df in tqdm(test_datasets_for_overlap_analysis.items()):
    df_f=filter_to_cols(df, target_cols)
    df_f.seq=df_f.seq.str.replace("$", "")
    df_f["seq_len"]=df_f.seq.apply(len)
    df_f=create_seq_allele_column(df_f)
    test_datasets_for_overlap_analysis[df_name]=df_f

100%|██████████| 10/10 [00:02<00:00,  4.75it/s]

CPU times: user 1.99 s, sys: 128 ms, total: 2.12 s
Wall time: 2.11 s


In [ ]:
# create seq allele for all
for df_name, df in tqdm(test_datasets_for_overlap_analysis.items()):
    df=create_seq_allele_column(df)
    test_datasets_for_overlap_analysis[df_name]=df

100%|██████████| 10/10 [00:00<00:00, 13.42it/s]


### Load model-specific train

 - We are primarily concerned with train sets for each model, not neceessarily test sets as it is not fully known whether test sets were integrated for each model.

#### HLApollo

 - Data downloaded from Zenodo (Version v1): https://zenodo.org/records/7951718. Use specifically BM1 for training.

In [ ]:
%%time
# bm1 test dat; train is more expansive, need to get all data files
bm1_test=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/test/BM1_test.csv")

# add sa ma col and process
bm1_test["sa_ma"]=["MA" if "_" in a else "SA" for a in bm1_test.allele]
bm1_test.rename(columns={"peptide": "seq", "el": "label"}, inplace=True)

bm1_test_sa=bm1_test.query("sa_ma=='SA'")
bm1_test_sa.allele=bm1_test_sa.allele.str.replace("*", "").str.replace(":", "")

print_data_stats(bm1_test_sa)
print(bm1_test_sa.allele.nunique())

Total: 1,412,200
Pos: 14,122 | 1.00%
Neg: 1,398,078 | 99.00%
N alleles: 107
107
CPU times: user 8.13 s, sys: 1.03 s, total: 9.16 s
Wall time: 9.18 s


In [ ]:
# train files
train_files=os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/train/")
print(len(train_files), train_files)

21 ['BM1_negative_train_set_14.csv', 'BM1_negative_train_set_18.csv', 'BM1_negative_train_set_11.csv', 'BM1_negative_train_set_04.csv', 'BM1_negative_train_set_15.csv', 'BM1_negative_train_set_16.csv', 'BM1_positive_train.csv', 'BM1_negative_train_set_07.csv', 'BM1_negative_train_set_01.csv', 'BM1_negative_train_set_20.csv', 'BM1_negative_train_set_12.csv', 'BM1_negative_train_set_09.csv', 'BM1_negative_train_set_05.csv', 'BM1_negative_train_set_08.csv', 'BM1_negative_train_set_10.csv', 'BM1_negative_train_set_13.csv', 'BM1_negative_train_set_03.csv', 'BM1_negative_train_set_02.csv', 'BM1_negative_train_set_06.csv', 'BM1_negative_train_set_17.csv', 'BM1_negative_train_set_19.csv']


In [ ]:
%%time
# aggregate all train
bm1_train_dfs={}
for f in tqdm(train_files):
    if f!="BM1_train_processed.csv":
        bm1_train_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/train/{f}")
        bm1_train_df["filename"]=f
        bm1_train_df["sa_ma"]=["MA" if "_" in a else "SA" for a in bm1_train_df.allele]
        bm1_train_dfs[f]=bm1_train_df
bm1_train=pd.concat(bm1_train_dfs.values())

  0%|          | 0/21 [00:00<?, ?it/s]

100%|██████████| 21/21 [00:39<00:00,  1.86s/it]


CPU times: user 37.4 s, sys: 4.64 s, total: 42 s
Wall time: 42.4 s


In [ ]:
# process
bm1_train.rename(columns={"peptide": "seq", "el": "label"}, inplace=True)

bm1_train_sa=bm1_train.query("sa_ma=='SA'")
bm1_train_sa.allele=bm1_train_sa.allele.str.replace("*", "").str.replace(":", "")

bm1_train_ma=bm1_train.query("sa_ma!='SA'")

In [ ]:
# expand ma
def expand_allele_underscore(df, allele_col="allele", id_col="peptide_id"):
    work = df.reset_index(drop=True).copy()
    work[id_col] = work.index
    work[allele_col] = work[allele_col].astype(str).str.split("_")
    out = work.explode(allele_col, ignore_index=True)
    out[allele_col] = out[allele_col].str.strip()
    out = out[out[allele_col] != ""]
    return out.reset_index(drop=True)

bm1_train_ma_exp=expand_allele_underscore(bm1_train_ma, allele_col="allele", id_col="peptide_id")

In [ ]:
# format allele
bm1_train_ma_exp.allele=bm1_train_ma_exp.allele.str.replace("*", "").str.replace(":", "")

In [ ]:
# merge
hlapollo_train=pd.concat([bm1_train_sa, bm1_train_ma_exp])

In [ ]:
"""
DFs:

 - bm1_train_sa
 - bm1_train_ma_exp
 - hlapollo_train
"""

'\nDFs:\n\n - bm1_train_sa\n - bm1_train_ma_exp\n - hlapollo_train\n'

##### MixMHCpred3.0

In [ ]:
# import
mixmhcpred_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/mixmhcpred/13073_2025_1450_MOESM2_ESM.txt", sep="\t")

In [ ]:
# rename cols and  reorder
mixmhcpred_train.columns=["allele", "seq", "label"]
mixmhcpred_train=mixmhcpred_train[["seq", "allele", "label"]]

In [ ]:
"""
DFs:

 - mixmhcpred_train
"""

'\nDFs:\n\n - mixmhcpred_train\n'

##### MUNIS

In [ ]:
# import
munis_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/munis/el_train.csv", index_col=0)

munis_test=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/munis/el_test.csv", index_col=0)

In [ ]:
# process
munis_train.rename(columns={"mhc": "allele", "pep": "seq", "left": "n_flank", "right": "c_flank"}, inplace=True)
munis_test.rename(columns={"mhc": "allele", "pep": "seq", "left": "n_flank", "right": "c_flank"}, inplace=True)

munis_train=munis_train[["seq", "allele", "label", "n_flank", "c_flank"]]
munis_test=munis_test[["seq", "allele", "label", "n_flank", "c_flank"]]

munis_train.allele=munis_train.allele.str.replace("HLA-", "").str.replace(":", "")
munis_test.allele=munis_test.allele.str.replace("HLA-", "").str.replace(":", "")

In [ ]:
"""
DFs:

 - munis_train
"""

'\nDFs:\n\n - munis_train\n'

##### BigMHC

In [ ]:
# import
bigmhc_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/bigmhc/el_train.csv")

bigmhc_val=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/bigmhc/el_val.csv")

bigmhc_test=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/bigmhc/el_test.csv")

In [ ]:
# process
bigmhc_train_val=pd.concat([bigmhc_train, bigmhc_val])

bigmhc_train_val.columns=["allele", "seq", "label"]
bigmhc_test=bigmhc_test[["mhc", "pep", "tgt"]]
bigmhc_test.columns=["allele", "seq", "label"]

bigmhc_train_val=bigmhc_train_val[["seq", "allele", "label"]]
bigmhc_test=bigmhc_test[["seq", "allele", "label"]]

bigmhc_train_val.allele=bigmhc_train_val.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
bigmhc_test.allele=bigmhc_test.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

In [ ]:
"""
DFs:

 - bigmhc_train_val
"""

'\nDFs:\n\n - bigmhc_train_val\n'

##### MHCFlurry2.0

In [ ]:
# import
mhcflurry_s3=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/MHCFlurry/Data_S3.csv")
mhcflurry_s5=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/MHCFlurry/Data_S5.csv")
mhcflurry_s6=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/MHCFlurry/Data_S6.csv")

In [ ]:
def expand_hla(df, hla_col="hla", id_col="peptide_id"):
    work = df.reset_index(drop=True).copy()
    work[id_col] = work.index
    work[hla_col] = work[hla_col].astype(str).str.split()
    out = work.explode(hla_col, ignore_index=True)
    out[hla_col] = out[hla_col].str.strip()
    out = out[out[hla_col] != ""]
    return out.reset_index(drop=True)

mhcflurry_s6_exp=expand_hla(mhcflurry_s6, hla_col="hla", id_col="peptide_id")

In [ ]:
# process
def expand_hla(df, hla_col="hla", id_col="peptide_id"):
    work = df.reset_index(drop=True).copy()
    work[id_col] = work.index
    work[hla_col] = work[hla_col].astype(str).str.split()
    out = work.explode(hla_col, ignore_index=True)
    out[hla_col] = out[hla_col].str.strip()
    out = out[out[hla_col] != ""]
    return out.reset_index(drop=True)

mhcflurry_s6_exp=expand_hla(mhcflurry_s6, hla_col="hla", id_col="peptide_id")

mhcflurry_s3.rename(columns={"peptide": "seq"}, inplace=True)
mhcflurry_s5.rename(columns={"peptide": "seq", "hit": "label", "hla": "allele"}, inplace=True)
mhcflurry_s6_exp.rename(columns={"peptide": "seq", "hit": "label", "hla": "allele"}, inplace=True)

mhcflurry_s3.allele=mhcflurry_s3.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
mhcflurry_s5.allele=mhcflurry_s5.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
mhcflurry_s6_exp.allele=mhcflurry_s6_exp.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

mhcflurry_s5['label'] = mhcflurry_s5['label'].astype(int)

In [ ]:
# merge
mhcflurry_train=pd.concat([mhcflurry_s3, mhcflurry_s5, mhcflurry_s6_exp])

In [ ]:
"""
DFs:

 - mhcflurry_s3
 - mhcflurry_s5
 - mhcflurry_s6_exp
 - mhcflurry_train
"""

'\nDFs:\n\n - mhcflurry_s3\n - mhcflurry_s5\n - mhcflurry_s6_exp\n - mhcflurry_train\n'

##### NetMHCpan4.1 EL/BA

In [ ]:
# read allele list
import re

def parse_file_to_dict(file_path):
   result = {}
   
   with open(file_path, 'r') as f:
       for line in f:
           line = line.strip()
           if not line:
               continue
           parts = re.split(r'\s+', line, 1)
           if len(parts) == 2:
               key = parts[0]
               values = [val.strip() for val in parts[1].split(',')]
               result[key] = values
   
   return result

# get allelelist
nmp_allelelist = parse_file_to_dict(f'{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/allelelist')

# isolate single alleles and hla alleles
nmp_allelelist_sa = {k: v[0] for k, v in nmp_allelelist.items() if len(v) == 1}
nmp_allelelist_ma = {k: v for k, v in nmp_allelelist.items() if len(v) > 1}

In [ ]:
# read in pseudosequences
mhc_pseudosequences_nmp = pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/MHC_pseudo.dat", sep=r"\s+", engine="python", header=None)
mhc_pseudosequences_nmp.columns=["allele", "seq"]

In [ ]:
# get raw data from their folder of info
nmp_files=os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41")

# ba data
nmp_ba_data=pd.DataFrame()
for f in tqdm(nmp_files):
    if f.endswith("ba"):
        df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/{f}", sep=" ", header=None)
        nmp_ba_data=pd.concat([nmp_ba_data, df], axis=0)

nmp_ba_data.columns=["seq", "ba_value", "allele"]

# el data
nmp_el_data=pd.DataFrame()
for f in tqdm(nmp_files):
    if f.endswith("el"):
        df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/{f}", sep=" ", header=None)
        nmp_el_data=pd.concat([nmp_el_data, df], axis=0)

nmp_el_data.columns=["seq", "label", "allele"]

  0%|          | 0/12 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:06<00:00,  1.95it/s]


In [ ]:
# split sa and ma for el and expand ma

def expand_allele_groups(df, allele_map, allele_col="allele", group_col="allele_group", id_col="peptide_id", drop_unmapped=False):
    work = df.reset_index(drop=True).copy()
    work[id_col] = work.index
    work = work.rename(columns={allele_col: group_col})
    work[allele_col] = work[group_col].map(allele_map)
    if drop_unmapped:
        work = work[work[allele_col].notna()]
    else:
        work[allele_col] = work.apply(lambda r: r[allele_col] if isinstance(r[allele_col], list) else [r[group_col]], axis=1)
    out = work.explode(allele_col, ignore_index=True)
    out[allele_col] = out[allele_col].astype(str).str.strip()
    out = out[out[allele_col] != ""]
    return out.reset_index(drop=True)

nmp_el_data_sa=nmp_el_data[nmp_el_data.allele.isin(list(nmp_allelelist_sa.keys()))]
nmp_el_data_ma=nmp_el_data[nmp_el_data.allele.isin(list(nmp_allelelist_ma.keys()))]
nmp_el_data_ma_exp=expand_allele_groups(nmp_el_data_ma, nmp_allelelist_ma, allele_col="allele", group_col="allele_group", id_col="peptide_id", drop_unmapped=False)

In [ ]:
# process
nmp_el_data_sa.allele=nmp_el_data_sa.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
nmp_el_data_ma_exp.allele=nmp_el_data_ma_exp.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
nmp_ba_data.allele=nmp_ba_data.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

In [ ]:
# merge
nmp41_train=pd.concat([nmp_el_data_sa, nmp_el_data_ma_exp, nmp_ba_data])

In [ ]:
"""
DFs:

 - nmp_el_data_sa
 - nmp_el_data_ma_exp
 - nmp_ba_data
"""

'\nDFs:\n\n - nmp_el_data_sa\n - nmp_el_data_ma_exp\n - nmp_ba_data\n'

##### NetMHCpan4.2 EL/BA + NE + PS

In [ ]:
# base sets
nmp42_el_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/netmhcpan/el_train.txt", sep=" ", header=None)
nmp42_ba_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/netmhcpan/ba_train.txt", sep=" ", header=None)

# ps sets
nmp42_iedb_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/netmhcpan/iedb_train.csv")

# ne sets
nmp42_cedar_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/netmhcpan/cedar_train.csv")

# metadata
nmp42_multiallele_metadata=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/netmhcpan/multiallele_metadata.csv")

In [ ]:
import pandas as pd

def build_id_to_allele_dict(df, id_col="Dataset ID", allele_col="MHC alleles expressed in cell line"):
    out = {}
    for _, row in df.iterrows():
        val = row[allele_col]
        if pd.isna(val):
            continue
        alleles = [a.strip() for a in str(val).split(",") if a.strip()]
        if alleles:
            out[row[id_col]] = alleles
    return out

nmp42_id_to_allele_set=build_id_to_allele_dict(nmp42_multiallele_metadata, id_col="Dataset ID", allele_col="MHC alleles expressed in cell line")

# isolate single alleles and hla alleles
nmp42_allelelist_sa = {k: v[0] for k, v in nmp42_id_to_allele_set.items() if len(v) == 1}
nmp42_allelelist_ma = {k: v for k, v in nmp42_id_to_allele_set.items() if len(v) > 1}

In [ ]:
# add cols
nmp42_el_train.columns=["seq", "label", "allele", "context"]
nmp42_ba_train.columns=["seq", "ba_value", "allele", "context"]
nmp42_iedb_train.columns=["seq", "label", "allele", "context"]
nmp42_cedar_train.columns=["seq", "label", "allele", "context"]

In [ ]:
# split into sa and ma
nmp42_el_train_sa=nmp42_el_train[nmp42_el_train.allele.isin(list(nmp42_allelelist_sa.keys()))]
nmp42_el_train_ma=nmp42_el_train[nmp42_el_train.allele.isin(list(nmp42_allelelist_ma.keys()))]
nmp42_el_train_ma_exp=expand_allele_groups(nmp42_el_train_ma, nmp42_allelelist_ma, allele_col="allele", group_col="allele_group", id_col="peptide_id", drop_unmapped=False)

In [ ]:
# map sa as well
nmp42_el_train_sa["allele"]=nmp42_el_train_sa.allele.map(nmp42_allelelist_sa)

In [ ]:
# process alleles
nmp42_el_train_sa.allele=nmp42_el_train_sa.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
nmp42_el_train_ma_exp.allele=nmp42_el_train_ma_exp.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
nmp42_ba_train.allele=nmp42_ba_train.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
nmp42_iedb_train.allele=nmp42_iedb_train.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")
nmp42_cedar_train.allele=nmp42_cedar_train.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

In [ ]:
# merge for ps and ne separate; both have el sa/ma, and ba, but ps has iedb and ne has cedar
nmp42_ps_train=pd.concat([nmp42_el_train_sa, nmp42_el_train_ma_exp, nmp42_ba_train, nmp42_iedb_train])
nmp42_ne_train=pd.concat([nmp42_el_train_sa, nmp42_el_train_ma_exp, nmp42_ba_train, nmp42_cedar_train])

In [ ]:
"""
DFs:

 - nmp42_el_train_sa
 - nmp42_el_train_ma_exp
 - nmp42_ba_train
 - nmp42_ps_train
 - nmp42_ne_train
"""

'\nDFs:\n\n - nmp42_el_train_sa\n - nmp42_el_train_ma_exp\n - nmp42_ba_train\n - nmp42_ps_train\n - nmp42_ne_train\n'

##### CapsNet-MHC Anthem

In [ ]:
# import
capsnetmhc_anthem_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/capsnetmhc_anthem/train_data.csv")

In [ ]:
# process
capsnetmhc_anthem_train.columns=["allele", "seq", "label", "seq_len"]
capsnetmhc_anthem_train.allele=capsnetmhc_anthem_train.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

In [ ]:
"""
DFs:

 - capsnetmhc_anthem_train
"""

'\nDFs:\n\n - capsnetmhc_anthem_train\n'

##### CapsNet-MHC IEDB

In [ ]:
# import
capsnetmhc_iedb_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/capsnetmhc_iedb/training_set.csv")

In [ ]:
# process
capsnetmhc_iedb_train.rename(columns={"sequence": "seq", "mhc": "allele", "peptide_length": "seq_len"}, inplace=True)
capsnetmhc_iedb_train.allele=capsnetmhc_iedb_train.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

In [ ]:
"""
DFs:

 - capsnetmhc_iedb_train
"""

'\nDFs:\n\n - capsnetmhc_iedb_train\n'

##### DeepAttentionPan

In [ ]:
# import
deepattentionpan_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/deepattentionpan/training_set.csv")

In [ ]:
# process
deepattentionpan_train.rename(columns={"sequence": "seq", "mhc": "allele", "peptide_length": "seq_len"}, inplace=True)
deepattentionpan_train.allele=deepattentionpan_train.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

In [ ]:
"""
DFs:

 - deepattentionpan_train
"""

'\nDFs:\n\n - deepattentionpan_train\n'

##### TransPHLA

In [ ]:
# import
transphla_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Model_Specific_Training_Sets/revisions_datasets/transphla/train_set.csv", index_col=0)

In [ ]:
# process
transphla_train.rename(columns={"peptide": "seq", "HLA": "allele", "length": "seq_len"}, inplace=True)
transphla_train.allele=transphla_train.allele.str.replace("HLA-", "").str.replace(":", "").str.replace("*", "")

In [ ]:
"""
DFs:

 - transphla_train
"""

'\nDFs:\n\n - transphla_train\n'

### Leakage analysis

In [ ]:
# aggregate train sources per model
train_sets_per_model={
    "mhcprime": ms_train_val_data,
    "hlapollo": hlapollo_train,
    "mixmhcpred": mixmhcpred_train,
    "munis": munis_train,
    "bigmhc": bigmhc_train_val,
    "mhcflurry": mhcflurry_train,
    "nmp41": nmp41_train,
    "nmp42_ps": nmp42_ps_train,
    "nmp42_ne": nmp42_ne_train,
    "capsnetmhc_anthem": capsnetmhc_anthem_train,
    "capsnetmhc_iedb": capsnetmhc_iedb_train,
    "deepattentionpan": deepattentionpan_train,
    "transphla": transphla_train,
}

# model tag to model name formal
model_tag_to_model_name = {
    "mhcprime": "MHCPrime",
    "hlapollo": "HLApollo",
    "mixmhcpred": "MixMHCpred3.0",
    "munis": "MUNIS",
    "bigmhc": "BigMHC EL",
    "mhcflurry": "MHCFlurry2.0",
    "nmp41": "NetMHCpan4.1 EL/BA",
    "nmp42_ps": "NetMHCpan4.2 PS",
    "nmp42_ne": "NetMHCpan4.2 NE",
    "capsnetmhc_anthem": "CapsNet-MHC Anthem",
    "capsnetmhc_iedb": "CapsNet-MHC IEDB",
    "deepattentionpan": "DeepAttentionPan",
    "transphla": "TransPHLA",
}

overlap_table_columns={
    "Model":"Model",
    "overlap_type":"Overlap type",
    "ms":"MS",
    "yd":"YD",
    "episcan_r9":"Epi. R9",
    "episcan_rt":"Epi. RT",
    "episcan_cov":"Epi. CoV",
    "ba_cls":"BA",    
    "escape_cls": "ESCAPE-seq",
}

overlap_type_order=["peptide-allele", "peptide", "allele"]

In [ ]:
# create seq allele for all
for df_name, df in tqdm(train_sets_per_model.items()):
    df=create_seq_allele_column(df)
    train_sets_per_model[df_name]=df

100%|██████████| 13/13 [01:11<00:00,  5.53s/it]


In [ ]:
%%time
# train test overlaps
model_train_test_overlap_df=compute_train_test_overlaps_fast(
    test_datasets_for_overlap_analysis,
    train_sets_per_model,
    output="percent",
    allele_col="allele",
    seq_col="seq",
    seq_allele_col="seq_allele",
    test_label_subset="both"
)

Precomputing unique benchmark entities (test_label_subset='both')...


Test datasets:   0%|          | 0/10 [00:00<?, ?it/s]

Test datasets: 100%|██████████| 10/10 [00:02<00:00,  3.71it/s]


Building benchmark unions...
  allele: 147 unique benchmark values
  peptide: 3,321,218 unique benchmark values
  peptide-allele: 3,991,878 unique benchmark values
Scanning training datasets...


Training scans: 100%|██████████| 39/39 [02:50<00:00,  4.37s/it, transphla | peptide-allele]        

CPU times: user 2min 43s, sys: 13.4 s, total: 2min 56s
Wall time: 2min 56s


In [ ]:
%%time
# train test overlaps (pos test)
model_train_test_overlap_df_pos=compute_train_test_overlaps_fast(
    test_datasets_for_overlap_analysis,
    train_sets_per_model,
    output="percent",
    allele_col="allele",
    seq_col="seq",
    seq_allele_col="seq_allele",
    test_label_subset="pos"
)

Precomputing unique benchmark entities (test_label_subset='pos')...


Test datasets: 100%|██████████| 10/10 [00:00<00:00, 188.05it/s]


Building benchmark unions...
  allele: 147 unique benchmark values
  peptide: 54,772 unique benchmark values
  peptide-allele: 82,239 unique benchmark values
Scanning training datasets...


Training scans:   0%|          | 0/39 [00:00<?, ?it/s]

Training scans: 100%|██████████| 39/39 [01:05<00:00,  1.68s/it, transphla | peptide-allele]        

CPU times: user 55.9 s, sys: 10 s, total: 1min 5s
Wall time: 1min 5s


In [ ]:
# save for plotting
# model_train_test_overlap_df.to_csv(f"{DATA_DIR}/Revisions_Data/model_train_test_overlap_df.csv")
# model_train_test_overlap_df_pos.to_csv(f"{DATA_DIR}/Revisions_Data/model_train_test_overlap_df_pos.csv")

### Generate train/validation/test supplementary data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
from src.general_utils import *
from src.model_utils import *

# ============================================================
# PROJECT PATH
# ============================================================

# If the notebook is launched from the project root, this is sufficient.
# Otherwise set PROJECT_ROOT manually once.
PROJECT_ROOT=Path.cwd().resolve().parent

if not (PROJECT_ROOT/"Data").exists():
    raise FileNotFoundError(
        f"Could not find {PROJECT_ROOT/'Data'}.\n"
        "Set PROJECT_ROOT to the same project root used by the training scripts."
    )

DATA_DIR=PROJECT_ROOT/"Data"


# ============================================================
# COUNTING HELPERS
# ============================================================

def _format_pos_neg_ratio(n_pos,n_neg):
    """Format positive:negative ratio in an interpretable 1:X form."""
    n_pos=int(n_pos); n_neg=int(n_neg)

    if n_pos==0 and n_neg==0:
        return "NA"
    if n_pos==0:
        return "0:1"
    if n_neg==0:
        return "1:0"

    ratio=n_neg/n_pos

    # Keep integers clean, otherwise use 2 decimals.
    if np.isclose(ratio,round(ratio)):
        return f"1:{int(round(ratio))}"

    return f"1:{ratio:.2f}"


def count_peptide_alleles_by_allele(df,allele_col="allele",label_col="label"):
    """
    Count positive/negative peptide-allele rows per allele.

    This intentionally counts dataframe rows, matching the number of
    peptide-allele observations present in the actual split.
    """
    if allele_col not in df.columns:
        raise KeyError(f"Missing allele column: {allele_col}")
    if label_col not in df.columns:
        raise KeyError(f"Missing label column: {label_col}")

    labels=pd.to_numeric(df[label_col],errors="coerce")

    unexpected=labels[~labels.isin([0,1]) & labels.notna()].unique()
    if len(unexpected):
        raise ValueError(
            f"Found non-binary labels in {label_col}: {unexpected[:10]}. "
            "Counts assume label 1 = positive and label 0 = negative."
        )

    work=pd.DataFrame({
        "allele":df[allele_col].astype(str).to_numpy(),
        "label":labels.to_numpy(),
    })

    rows=[]

    for allele,g in work.groupby("allele",sort=True):
        n_pos=int((g["label"]==1).sum())
        n_neg=int((g["label"]==0).sum())

        if n_pos+n_neg!=len(g):
            raise ValueError(
                f"{allele}: {len(g)-(n_pos+n_neg)} rows have missing/non-binary labels."
            )

        rows.append({
            "Allele":allele,
            "Positive peptide-alleles":n_pos,
            "Negative peptide-alleles":n_neg,
            "Total peptide-alleles":n_pos+n_neg,
            "Positive to negative ratio":_format_pos_neg_ratio(n_pos,n_neg),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "Allele",
            "Positive peptide-alleles",
            "Negative peptide-alleles",
            "Total peptide-alleles",
            "Positive to negative ratio",
        ],
    )


def summarize_count_tables(count_tables):
    """
    Create one overall summary row for every count table.
    """
    rows=[]

    for name,df in count_tables.items():
        n_pos=int(df["Positive peptide-alleles"].sum())
        n_neg=int(df["Negative peptide-alleles"].sum())

        rows.append({
            "Split":name,
            "Total alleles":int(df["Allele"].nunique()),
            "Positive peptide-alleles":n_pos,
            "Negative peptide-alleles":n_neg,
            "Total peptide-alleles":n_pos+n_neg,
            "Positive to negative ratio":_format_pos_neg_ratio(n_pos,n_neg),
        })

    return pd.DataFrame(rows)


# ============================================================
# EXACT MATCHING HELPERS USED FOR COUNT GENERATION
# ============================================================

def _add_seq_len_no_pad(df):
    df=df.copy()
    df["seq_len"]=df.seq.str.replace("$","").apply(len)
    return df


def _build_yd_train(ms_train,ft_train):
    """
    Exact YD train-set membership logic used by the scripts.
    """
    ms=ms_train.copy()
    ft=ft_train.copy()

    ms["domain_id"]=0
    ft["domain_id"]=1

    train_set=pd.concat([ms,ft],ignore_index=True)
    train_set["seq_len"]=train_set.seq.str.replace("$","").apply(len)

    train_set=(
        train_set[
            train_set.allele.isin(ft.allele.unique())
        ]
        .query("seq_len == 9")
        .reset_index(drop=True)
    )

    assay_train=train_set.query("domain_id == 1").copy()
    matched_ms_train=train_set.query("domain_id == 0").copy()

    return assay_train,matched_ms_train


def _build_yd_matched_ms(ms_df,assay_df):
    """
    Exact YD MS validation/test matching logic.
    """
    ms=_add_seq_len_no_pad(ms_df)

    return (
        ms[
            ms.allele.isin(assay_df.allele.unique())
        ]
        .query("seq_len == 9")
        .copy()
        .reset_index(drop=True)
    )


EPISCAN_ISO_ALLELES=["A0201","A0301","B0801","B5701"]


def _build_episcan_train(ms_train,r9_train,rt_train,mode):
    """
    Exact EpiScan training membership logic.

    mode="r9":
        R9 positives/negatives + RT positives only

    mode="r9rt":
        R9 positives/negatives + RT positives/negatives
    """
    if mode not in {"r9","r9rt"}:
        raise ValueError("mode must be 'r9' or 'r9rt'.")

    ms=ms_train.copy()
    r9=r9_train.copy()
    rt=rt_train.copy()

    ms["domain_id"]=0
    r9["domain_id"]=1
    rt["domain_id"]=1

    if mode=="r9":
        ft_train=pd.concat(
            [r9,rt.query("label == 1")],
            ignore_index=True,
        )
    else:
        ft_train=pd.concat(
            [r9,rt],
            ignore_index=True,
        )

    train_set=pd.concat(
        [ms,ft_train],
        ignore_index=True,
    )

    # Preserve the exact EpiScan implementation from your scripts.
    train_set["seq_len"]=train_set.seq.apply(len)

    train_set=(
        train_set[
            train_set.allele.isin(EPISCAN_ISO_ALLELES)
        ]
        .query("seq_len == 9")
        .reset_index(drop=True)
    )

    assay_train=train_set.query("domain_id == 1").copy()
    matched_ms_train=train_set.query("domain_id == 0").copy()

    return assay_train,matched_ms_train


def _build_episcan_matched_ms(ms_df):
    """
    Exact EpiScan MS validation/test matching from the scripts:
    fixed 4 alleles and peptide length 9.
    """
    ms=_add_seq_len_no_pad(ms_df)

    return (
        ms[
            ms.allele.isin(EPISCAN_ISO_ALLELES)
        ]
        .query("seq_len == 9")
        .copy()
        .reset_index(drop=True)
    )


def _build_pair_matched_train(ms_train,ft_train):
    """
    Exact BA/ESCAPE-seq joint training matching:
    concatenate domain 0/1, then call the same project helper used in training.
    """
    ms=ms_train.copy()
    ft=ft_train.copy()

    ms["domain_id"]=0
    ft["domain_id"]=1

    train_set=pd.concat(
        [ms,ft],
        ignore_index=True,
    )

    train_set["seq_len"]=(
        train_set.seq
        .str.replace("$","")
        .apply(len)
    )

    train_set=filter_domain0_to_domain1_pairs(
        train_set,
        allele_col="allele",
        seq_len_col="seq_len",
        domain_col="domain_id",
    ).reset_index(drop=True)

    assay_train=train_set.query("domain_id == 1").copy()
    matched_ms_train=train_set.query("domain_id == 0").copy()

    return assay_train,matched_ms_train


def _build_pair_matched_ms(ms_df,assay_df):
    """
    Exact BA/ESCAPE-seq validation/test MS matching.
    """
    ms=_add_seq_len_no_pad(ms_df)
    assay=_add_seq_len_no_pad(assay_df)

    return filter_to_shared_pairs(
        ms,
        assay,
        allele_col="allele",
        seq_len_col="seq_len",
    ).copy().reset_index(drop=True)


# ============================================================
# MAIN BUILDER
# ============================================================

def build_primary_dataset_count_tables(DATA_DIR,verbose=True):
    """
    Reconstruct the primary + 4 CV train/validation splits and the single
    fixed test split for YD, EpiScan, BA, and cleaned ESCAPE-seq.

    Returns
    -------
    count_tables : dict[str, pd.DataFrame]
        Per-allele count tables.

    summary_df : pd.DataFrame
        Overall totals for every dictionary entry.
    """

    DATA_DIR=Path(DATA_DIR)

    # --------------------------------------------------------
    # PATHS - copied from the actual training scripts
    # --------------------------------------------------------
    ms_dir=DATA_DIR/"Training_Datasets/Processed/MS_Train_Test_Sets"
    yd_dir=DATA_DIR/"Training_Datasets/Processed/Holec_et_al_2025"
    epi_dir=DATA_DIR/"Training_Datasets/Processed/EpiScan"
    ba_dir=DATA_DIR/"Training_Datasets/Processed/BA"
    esc_dir=DATA_DIR/"Training_Datasets/Processed/ESCAPE_seq"

    # --------------------------------------------------------
    # LOAD COMMON MS ONCE
    # --------------------------------------------------------
    if verbose:
        print("Loading full MS datasets once...")

    ms_train_base=pd.read_csv(
        ms_dir/"ms_train_data_remaining_minimal.csv",
        index_col=0,
    )
    ms_val_base=pd.read_csv(
        ms_dir/"ms_val_data.csv",
        index_col=0,
    )
    ms_test_base=pd.read_csv(
        ms_dir/"ms_test_data.csv",
        index_col=0,
    )

    # --------------------------------------------------------
    # LOAD CV SPLIT DICTIONARIES ONCE
    # --------------------------------------------------------
    if verbose:
        print("Loading CV split dictionaries...")

    yd_cv=load_dict_pickle(
        str(yd_dir/"holec_train_val_sets.pkl")
    )

    epi_r9_cv=load_dict_pickle(
        str(epi_dir/"episcan_r9_train_val_sets.pkl")
    )
    epi_rt_cv=load_dict_pickle(
        str(epi_dir/"episcan_rt_train_val_sets.pkl")
    )

    ba_cv=load_dict_pickle(
        str(ba_dir/"ba_con_train_val_sets.pkl")
    )

    escape_cv=load_dict_pickle(
        str(esc_dir/"escape_con_train_val_sets_cleaned.pkl")
    )

    for name,d in [
        ("YD",yd_cv),
        ("EpiScan R9",epi_r9_cv),
        ("EpiScan RT",epi_rt_cv),
        ("BA",ba_cv),
        ("ESCAPE-seq",escape_cv),
    ]:
        missing=[x for x in [1,2,3,4] if x not in d]
        if missing:
            raise KeyError(
                f"{name} CV pickle missing split IDs: {missing}. "
                f"Available keys: {sorted(d)}"
            )

    # --------------------------------------------------------
    # LOAD PRIMARY SPLITS
    # --------------------------------------------------------
    if verbose:
        print("Loading primary assay splits...")

    yd_primary_train=pd.read_csv(
        yd_dir/"holec_train_set_remaining.csv",
        index_col=0,
    )
    yd_primary_val=pd.read_csv(
        yd_dir/"holec_val_set.csv",
        index_col=0,
    )
    yd_test=pd.read_csv(
        yd_dir/"holec_test_set.csv",
        index_col=0,
    )

    epi_r9_primary_train=pd.read_csv(
        epi_dir/"episcan_r9_train_set_remaining.csv",
        index_col=0,
    )
    epi_rt_primary_train=pd.read_csv(
        epi_dir/"episcan_rt_train_set_remaining.csv",
        index_col=0,
    )
    epi_r9_primary_val=pd.read_csv(
        epi_dir/"episcan_r9_val_set.csv",
        index_col=0,
    )
    epi_rt_primary_val=pd.read_csv(
        epi_dir/"episcan_rt_val_set.csv",
        index_col=0,
    )
    epi_r9_test=pd.read_csv(
        epi_dir/"episcan_r9_test_set.csv",
        index_col=0,
    )
    epi_rt_test=pd.read_csv(
        epi_dir/"episcan_rt_test_set.csv",
        index_col=0,
    )

    ba_primary_train=pd.read_csv(
        ba_dir/"train_df_ba_for_updating_remaining.csv",
        index_col=0,
    )
    ba_primary_val=pd.read_csv(
        ba_dir/"ba_val_set.csv",
        index_col=0,
    )
    ba_test=pd.read_csv(
        ba_dir/"test_df_ba_for_updating.csv",
        index_col=0,
    )

    # CLEANED ESCAPE-seq versions used in the current scripts.
    escape_primary_train=pd.read_csv(
        esc_dir/"escape_con_train_set_remaining_cleaned.csv",
        index_col=0,
    )
    escape_primary_val=pd.read_csv(
        esc_dir/"escape_con_val_set_cleaned.csv",
        index_col=0,
    )
    escape_test=pd.read_csv(
        esc_dir/"escape_con_test_set_cleaned.csv",
        index_col=0,
    )

    count_tables={}

    def add(name,df):
        if name in count_tables:
            raise KeyError(f"Duplicate count-table key: {name}")

        table=count_peptide_alleles_by_allele(df)
        count_tables[name]=table

        if verbose:
            print(
                f"  {name}: "
                f"{table['Total peptide-alleles'].sum():,} rows | "
                f"{table['Allele'].nunique():,} alleles"
            )

    # ========================================================
    # FULL MS
    # ========================================================
    print("\nAdding full MS splits...")

    add("Full MS Train",ms_train_base)
    add("Full MS Validation",ms_val_base)
    add("Full MS Test",ms_test_base)

    # ========================================================
    # FIXED TEST SETS
    #
    # Only one test split exists. Do not repeat it for every CV split.
    # ========================================================
    print("\nAdding fixed test splits...")

    # YD
    yd_ms_test=_build_yd_matched_ms(
        ms_test_base.copy(),
        yd_test.copy(),
    )

    add("YD | Test | Assay",yd_test)
    add("YD | Test | Matched MS",yd_ms_test)

    # EpiScan
    epi_ms_test=_build_episcan_matched_ms(
        ms_test_base.copy()
    )

    add("EpiScan | R9 Test | Assay",epi_r9_test)
    add("EpiScan | R9 Test | Matched MS",epi_ms_test.copy())

    add("EpiScan | RT Test | Assay",epi_rt_test)
    add("EpiScan | RT Test | Matched MS",epi_ms_test.copy())

    # BA
    ba_ms_test=_build_pair_matched_ms(
        ms_test_base.copy(),
        ba_test.copy(),
    )

    add("BA | Test | Assay",ba_test)
    add("BA | Test | Matched MS",ba_ms_test)

    # ESCAPE-seq
    escape_ms_test=_build_pair_matched_ms(
        ms_test_base.copy(),
        escape_test.copy(),
    )

    add("ESCAPE-seq | Test | Assay",escape_test)
    add("ESCAPE-seq | Test | Matched MS",escape_ms_test)

    # ========================================================
    # PRIMARY + 4 ADDITIONAL TRAIN/VALIDATION SPLITS
    # ========================================================
    split_ids=["Primary",1,2,3,4]

    pbar=tqdm(
        total=len(split_ids)*4,
        desc="Building assay train/validation counts",
        leave=True,
    )

    for split_id in split_ids:
        split_name=(
            "Primary"
            if split_id=="Primary"
            else f"CV{split_id}"
        )

        # ====================================================
        # YD
        # ====================================================
        if split_id=="Primary":
            ft_train=yd_primary_train.copy()
            ft_val=yd_primary_val.copy()
        else:
            ft_train=yd_cv[split_id]["train"].copy()
            ft_val=yd_cv[split_id]["val"].copy()

        assay_train,ms_train_matched=_build_yd_train(
            ms_train_base.copy(),
            ft_train,
        )

        ms_val_matched=_build_yd_matched_ms(
            ms_val_base.copy(),
            ft_val,
        )

        add(f"YD | {split_name} | Assay Train",assay_train)
        add(f"YD | {split_name} | Matched MS Train",ms_train_matched)
        add(f"YD | {split_name} | Assay Validation",ft_val)
        add(f"YD | {split_name} | Matched MS Validation",ms_val_matched)

        pbar.update(1)

        # ====================================================
        # EPISCAN
        # ====================================================
        if split_id=="Primary":
            r9_train=epi_r9_primary_train.copy()
            rt_train=epi_rt_primary_train.copy()
            r9_val=epi_r9_primary_val.copy()
            rt_val=epi_rt_primary_val.copy()
        else:
            r9_train=epi_r9_cv[split_id]["train"].copy()
            rt_train=epi_rt_cv[split_id]["train"].copy()
            r9_val=epi_r9_cv[split_id]["val"].copy()
            rt_val=epi_rt_cv[split_id]["val"].copy()

        # R9 training regime:
        # R9 +/- + RT positives only
        epi_r9_train,epi_r9_ms_train=_build_episcan_train(
            ms_train_base.copy(),
            r9_train.copy(),
            rt_train.copy(),
            mode="r9",
        )

        # R9/RT training regime:
        # R9 +/- + RT +/-
        epi_r9rt_train,epi_r9rt_ms_train=_build_episcan_train(
            ms_train_base.copy(),
            r9_train.copy(),
            rt_train.copy(),
            mode="r9rt",
        )

        # Script uses the same fixed ISO allele/9-mer match for validation.
        epi_ms_val=_build_episcan_matched_ms(
            ms_val_base.copy()
        )

        add(
            f"EpiScan R9 | {split_name} | Assay Train (R9 + RT positives)",
            epi_r9_train,
        )
        add(
            f"EpiScan R9 | {split_name} | Matched MS Train",
            epi_r9_ms_train,
        )

        add(
            f"EpiScan R9/RT | {split_name} | Assay Train (R9 + RT all)",
            epi_r9rt_train,
        )
        add(
            f"EpiScan R9/RT | {split_name} | Matched MS Train",
            epi_r9rt_ms_train,
        )

        # Keep R9 and RT validation sets separate for reporting.
        add(
            f"EpiScan | {split_name} | R9 Validation | Assay",
            r9_val,
        )
        add(
            f"EpiScan | {split_name} | RT Validation | Assay",
            rt_val,
        )

        # Shared under the exact script matching logic.
        add(
            f"EpiScan | {split_name} | Matched MS Validation (shared R9/RT)",
            epi_ms_val,
        )

        pbar.update(1)

        # ====================================================
        # BA
        # ====================================================
        if split_id=="Primary":
            ft_train=ba_primary_train.copy()
            ft_val=ba_primary_val.copy()
        else:
            ft_train=ba_cv[split_id]["train"].copy()
            ft_val=ba_cv[split_id]["val"].copy()

        assay_train,ms_train_matched=_build_pair_matched_train(
            ms_train_base.copy(),
            ft_train,
        )

        ms_val_matched=_build_pair_matched_ms(
            ms_val_base.copy(),
            ft_val,
        )

        add(f"BA | {split_name} | Assay Train",assay_train)
        add(f"BA | {split_name} | Matched MS Train",ms_train_matched)
        add(f"BA | {split_name} | Assay Validation",ft_val)
        add(f"BA | {split_name} | Matched MS Validation",ms_val_matched)

        pbar.update(1)

        # ====================================================
        # ESCAPE-SEQ - CLEANED
        # ====================================================
        if split_id=="Primary":
            ft_train=escape_primary_train.copy()
            ft_val=escape_primary_val.copy()
        else:
            ft_train=escape_cv[split_id]["train"].copy()
            ft_val=escape_cv[split_id]["val"].copy()

        assay_train,ms_train_matched=_build_pair_matched_train(
            ms_train_base.copy(),
            ft_train,
        )

        ms_val_matched=_build_pair_matched_ms(
            ms_val_base.copy(),
            ft_val,
        )

        add(
            f"ESCAPE-seq | {split_name} | Assay Train",
            assay_train,
        )
        add(
            f"ESCAPE-seq | {split_name} | Matched MS Train",
            ms_train_matched,
        )
        add(
            f"ESCAPE-seq | {split_name} | Assay Validation",
            ft_val,
        )
        add(
            f"ESCAPE-seq | {split_name} | Matched MS Validation",
            ms_val_matched,
        )

        pbar.update(1)

    pbar.close()

    # ========================================================
    # FINAL SUMMARY
    # ========================================================
    summary_df=summarize_count_tables(
        count_tables
    )

    print(
        f"\nComplete: {len(count_tables):,} per-allele count tables generated."
    )
    print(
        f"Summary table: {len(summary_df):,} split-level rows."
    )

    return count_tables,summary_df

import numpy as np
import pandas as pd
from itertools import combinations


def report_cv_count_consistency(
    dataset_count_tables,
    tolerance=2,
    cv_splits=(1,2,3,4),
    print_per_allele_differences=False,
):
    """
    Compare CV1-CV4 count tables within each assay/split type.

    Checks both:
      1. overall table totals;
      2. per-allele positive, negative, and total peptide-allele counts.

    Classification
    --------------
    EXACT
        All CV splits have identical allele membership and identical
        per-allele positive/negative/total counts.

    WITHIN TOLERANCE
        Same allele membership, and every per-allele count differs by
        <= tolerance across CV splits.

    DIFFERENT
        Allele membership differs or at least one per-allele count differs
        by more than tolerance.

    Returns
    -------
    report_df : pd.DataFrame
        One row per logical assay/split group.

    details : dict
        Detailed pairwise/per-allele comparison information for each group.
    """

    # ============================================================
    # Exact logical groups created by build_primary_dataset_count_tables
    # ============================================================
    groups={
        # YD
        "YD | Assay Train":
            lambda cv:f"YD | CV{cv} | Assay Train",

        "YD | Matched MS Train":
            lambda cv:f"YD | CV{cv} | Matched MS Train",

        "YD | Assay Validation":
            lambda cv:f"YD | CV{cv} | Assay Validation",

        "YD | Matched MS Validation":
            lambda cv:f"YD | CV{cv} | Matched MS Validation",

        # EpiScan R9 training regime
        "EpiScan R9 | Assay Train":
            lambda cv:f"EpiScan R9 | CV{cv} | Assay Train (R9 + RT positives)",

        "EpiScan R9 | Matched MS Train":
            lambda cv:f"EpiScan R9 | CV{cv} | Matched MS Train",

        # EpiScan R9/RT training regime
        "EpiScan R9/RT | Assay Train":
            lambda cv:f"EpiScan R9/RT | CV{cv} | Assay Train (R9 + RT all)",

        "EpiScan R9/RT | Matched MS Train":
            lambda cv:f"EpiScan R9/RT | CV{cv} | Matched MS Train",

        # EpiScan validation remains separate R9 and RT
        "EpiScan R9 | Assay Validation":
            lambda cv:f"EpiScan | CV{cv} | R9 Validation | Assay",

        "EpiScan RT | Assay Validation":
            lambda cv:f"EpiScan | CV{cv} | RT Validation | Assay",

        "EpiScan R9/RT | Matched MS Validation":
            lambda cv:f"EpiScan | CV{cv} | Matched MS Validation (shared R9/RT)",

        # BA
        "BA | Assay Train":
            lambda cv:f"BA | CV{cv} | Assay Train",

        "BA | Matched MS Train":
            lambda cv:f"BA | CV{cv} | Matched MS Train",

        "BA | Assay Validation":
            lambda cv:f"BA | CV{cv} | Assay Validation",

        "BA | Matched MS Validation":
            lambda cv:f"BA | CV{cv} | Matched MS Validation",

        # ESCAPE-seq
        "ESCAPE-seq | Assay Train":
            lambda cv:f"ESCAPE-seq | CV{cv} | Assay Train",

        "ESCAPE-seq | Matched MS Train":
            lambda cv:f"ESCAPE-seq | CV{cv} | Matched MS Train",

        "ESCAPE-seq | Assay Validation":
            lambda cv:f"ESCAPE-seq | CV{cv} | Assay Validation",

        "ESCAPE-seq | Matched MS Validation":
            lambda cv:f"ESCAPE-seq | CV{cv} | Matched MS Validation",
    }

    count_cols=[
        "Positive peptide-alleles",
        "Negative peptide-alleles",
        "Total peptide-alleles",
    ]

    report_rows=[]
    details={}

    def _overall(df):
        pos=int(df["Positive peptide-alleles"].sum())
        neg=int(df["Negative peptide-alleles"].sum())
        total=int(df["Total peptide-alleles"].sum())

        return {
            "alleles":int(df["Allele"].nunique()),
            "positive":pos,
            "negative":neg,
            "total":total,
        }

    def _signature(df):
        return (
            df[
                ["Allele"]+count_cols
            ]
            .sort_values("Allele")
            .reset_index(drop=True)
        )

    for group_name,key_func in groups.items():

        keys={
            cv:key_func(cv)
            for cv in cv_splits
        }

        missing=[
            key
            for key in keys.values()
            if key not in dataset_count_tables
        ]

        if missing:
            print(
                f"\nWARNING: skipping {group_name}; "
                f"missing keys: {missing}"
            )
            continue

        tables={
            cv:_signature(
                dataset_count_tables[key]
            )
            for cv,key in keys.items()
        }

        totals={
            cv:_overall(
                dataset_count_tables[keys[cv]]
            )
            for cv in cv_splits
        }

        # --------------------------------------------------------
        # Overall ranges across CVs.
        # --------------------------------------------------------
        allele_vals=[
            totals[cv]["alleles"]
            for cv in cv_splits
        ]
        pos_vals=[
            totals[cv]["positive"]
            for cv in cv_splits
        ]
        neg_vals=[
            totals[cv]["negative"]
            for cv in cv_splits
        ]
        total_vals=[
            totals[cv]["total"]
            for cv in cv_splits
        ]

        # --------------------------------------------------------
        # Check allele membership first.
        # --------------------------------------------------------
        allele_sets={
            cv:set(tables[cv]["Allele"])
            for cv in cv_splits
        }

        same_alleles=all(
            allele_sets[cv]==allele_sets[cv_splits[0]]
            for cv in cv_splits[1:]
        )

        pairwise=[]
        max_abs_diff=0
        exact_all=True

        if same_alleles:
            indexed={
                cv:tables[cv].set_index("Allele")[count_cols]
                for cv in cv_splits
            }

            for cv1,cv2 in combinations(cv_splits,2):
                diff=(
                    indexed[cv1]
                    -indexed[cv2]
                ).abs()

                pair_max=int(
                    diff.to_numpy().max()
                ) if len(diff) else 0

                n_different=int(
                    (diff>0).any(axis=1).sum()
                )

                n_over_tol=int(
                    (diff>tolerance).any(axis=1).sum()
                )

                pairwise.append({
                    "cv1":cv1,
                    "cv2":cv2,
                    "max_abs_difference":pair_max,
                    "n_alleles_different":n_different,
                    "n_alleles_over_tolerance":n_over_tol,
                    "difference_table":diff,
                })

                max_abs_diff=max(
                    max_abs_diff,
                    pair_max,
                )

                if pair_max>0:
                    exact_all=False

        else:
            exact_all=False
            max_abs_diff=np.nan

        # --------------------------------------------------------
        # Classification.
        # --------------------------------------------------------
        if same_alleles and exact_all:
            status="EXACT"
            collapsible=True

        elif same_alleles and max_abs_diff<=tolerance:
            status=f"WITHIN {tolerance}"
            collapsible=True

        else:
            status="DIFFERENT"
            collapsible=False

        # --------------------------------------------------------
        # Exact-signature clusters.
        #
        # Useful if, e.g., CV1/CV2 are identical and CV3/CV4 identical.
        # --------------------------------------------------------
        signatures={}

        for cv in cv_splits:
            t=tables[cv]

            sig=tuple(
                map(
                    tuple,
                    t[
                        ["Allele"]+count_cols
                    ].to_numpy()
                )
            )

            signatures.setdefault(
                sig,
                []
            ).append(cv)

        exact_clusters=list(
            signatures.values()
        )

        cluster_string="; ".join(
            [
                ",".join(
                    f"CV{x}"
                    for x in cluster
                )
                for cluster in exact_clusters
            ]
        )

        report_rows.append({
            "Group":group_name,
            "Status":status,
            "Can collapse":collapsible,
            "Same allele set":same_alleles,
            "Exact CV clusters":cluster_string,
            "Alleles min":min(allele_vals),
            "Alleles max":max(allele_vals),
            "Positive min":min(pos_vals),
            "Positive max":max(pos_vals),
            "Negative min":min(neg_vals),
            "Negative max":max(neg_vals),
            "Total min":min(total_vals),
            "Total max":max(total_vals),
            "Maximum per-allele count difference":max_abs_diff,
        })

        details[group_name]={
            "keys":keys,
            "totals":totals,
            "same_alleles":same_alleles,
            "exact_clusters":exact_clusters,
            "pairwise":pairwise,
        }

    report_df=pd.DataFrame(
        report_rows
    )

    # ============================================================
    # PRINT REPORT
    # ============================================================
    print("\n"+"="*100)
    print("CV SPLIT COUNT CONSISTENCY REPORT")
    print("="*100)
    print(
        f"Tolerance: ±{tolerance} peptide-allele rows per allele/count category"
    )

    for _,row in report_df.iterrows():

        print("\n"+"-"*100)
        print(row["Group"])
        print("-"*100)

        print(
            f"Status: {row['Status']} | "
            f"Can collapse: {row['Can collapse']}"
        )

        print(
            f"Exact CV clusters: "
            f"{row['Exact CV clusters']}"
        )

        print(
            f"Alleles: "
            f"{row['Alleles min']:,}"
            if row["Alleles min"]==row["Alleles max"]
            else
            f"Alleles: {row['Alleles min']:,}-{row['Alleles max']:,}"
        )

        print(
            f"Positive peptide-alleles: "
            f"{row['Positive min']:,}"
            if row["Positive min"]==row["Positive max"]
            else
            f"Positive peptide-alleles: "
            f"{row['Positive min']:,}-{row['Positive max']:,}"
        )

        print(
            f"Negative peptide-alleles: "
            f"{row['Negative min']:,}"
            if row["Negative min"]==row["Negative max"]
            else
            f"Negative peptide-alleles: "
            f"{row['Negative min']:,}-{row['Negative max']:,}"
        )

        print(
            f"Total peptide-alleles: "
            f"{row['Total min']:,}"
            if row["Total min"]==row["Total max"]
            else
            f"Total peptide-alleles: "
            f"{row['Total min']:,}-{row['Total max']:,}"
        )

        print(
            f"Same allele membership: "
            f"{row['Same allele set']}"
        )

        print(
            f"Maximum per-allele count difference: "
            f"{row['Maximum per-allele count difference']}"
        )

        if (
            print_per_allele_differences
            and row["Status"]!="EXACT"
            and row["Same allele set"]
        ):
            for pair in details[row["Group"]]["pairwise"]:
                if pair["max_abs_difference"]==0:
                    continue

                print(
                    f"\n  CV{pair['cv1']} vs CV{pair['cv2']}: "
                    f"max diff={pair['max_abs_difference']}, "
                    f"alleles differing={pair['n_alleles_different']}, "
                    f"alleles > tolerance={pair['n_alleles_over_tolerance']}"
                )

    # ------------------------------------------------------------
    # Compact final grouping recommendation.
    # ------------------------------------------------------------
    print("\n"+"="*100)
    print("COLLAPSE SUMMARY")
    print("="*100)

    exact=report_df[
        report_df["Status"].eq("EXACT")
    ]

    near=report_df[
        report_df["Status"].str.startswith("WITHIN")
    ]

    different=report_df[
        report_df["Status"].eq("DIFFERENT")
    ]

    print(
        f"Exact across all CVs: "
        f"{len(exact)}/{len(report_df)} groups"
    )

    if len(exact):
        for x in exact["Group"]:
            print(f"  [EXACT] {x}")

    print(
        f"\nWithin tolerance but not exact: "
        f"{len(near)}/{len(report_df)} groups"
    )

    if len(near):
        for x in near["Group"]:
            print(f"  [NEAR]  {x}")

    print(
        f"\nMeaningfully different: "
        f"{len(different)}/{len(report_df)} groups"
    )

    if len(different):
        for x in different["Group"]:
            print(f"  [KEEP SEPARATE] {x}")

    return report_df,details

import pandas as pd


import pandas as pd


def collapse_dataset_count_tables(
    dataset_count_tables,
    collapse_all_cv_splits=True,
    representative_split="Primary",
    verbose=True,
):
    """
    Collapse dataset count tables for compact reporting.

    Parameters
    ----------
    dataset_count_tables : dict[str, pd.DataFrame]
        Output from build_primary_dataset_count_tables().

    collapse_all_cv_splits : bool
        If True:
            Collapse Primary + CV1 + CV2 + CV3 + CV4 into:
                "CV splits (n=5)"

        If False:
            Keep Primary separate and collapse only CV1-CV4 into:
                "Additional CV splits (n=4)"

    representative_split : {"Primary","CV1","CV2","CV3","CV4"}
        Which split's count table to use as the representative table when
        multiple splits are collapsed.

        This matters primarily for ESCAPE-seq, where train/validation counts
        differ by a few rows across splits. For groups with identical counts,
        the choice has no effect.

    verbose : bool
        Print source -> collapsed mappings.

    Returns
    -------
    collapsed_tables : dict[str, pd.DataFrame]
        Collapsed per-allele count tables.

    collapsed_summary : pd.DataFrame
        Overall totals for each collapsed table.
    """
    valid_representatives={"Primary","CV1","CV2","CV3","CV4"}

    if representative_split not in valid_representatives:
        raise ValueError(
            "representative_split must be one of "
            f"{sorted(valid_representatives)}."
        )

    collapsed_tables={}

    def add(new_key,source_key):
        if source_key not in dataset_count_tables:
            raise KeyError(f"Missing source table: {source_key}")

        if new_key in collapsed_tables:
            raise KeyError(f"Duplicate collapsed key: {new_key}")

        collapsed_tables[new_key]=dataset_count_tables[source_key].copy()

        if verbose:
            print(f"{new_key}")
            print(f"  <- {source_key}")

    # ============================================================
    # FULL MS - ALWAYS RETAIN
    # ============================================================
    add("Full MS Train","Full MS Train")
    add("Full MS Validation","Full MS Validation")
    add("Full MS Test","Full MS Test")

    # ============================================================
    # DEFINE COLLAPSE MODE
    # ============================================================
    if collapse_all_cv_splits:
        collapsed_label="CV splits (n=5)"
        source_split=representative_split
    else:
        collapsed_label="Additional CV splits (n=4)"

        if representative_split=="Primary":
            source_split="CV1"
        else:
            source_split=representative_split

    # ============================================================
    # HELPER FOR TRAIN/VAL SPLIT SOURCES
    # ============================================================
    def source(prefix,suffix):
        return f"{prefix} | {source_split} | {suffix}"

    # ============================================================
    # YD
    # ============================================================
    if collapse_all_cv_splits:
        add(
            f"YD | {collapsed_label} | Assay Train",
            source("YD","Assay Train"),
        )
        add(
            f"YD | {collapsed_label} | Matched MS Train",
            source("YD","Matched MS Train"),
        )
        add(
            f"YD | {collapsed_label} | Assay Validation",
            source("YD","Assay Validation"),
        )
        add(
            f"YD | {collapsed_label} | Matched MS Validation",
            source("YD","Matched MS Validation"),
        )

    else:
        add(
            "YD | Primary | Assay Train",
            "YD | Primary | Assay Train",
        )
        add(
            f"YD | {collapsed_label} | Assay Train",
            f"YD | {source_split} | Assay Train",
        )

        add(
            "YD | Primary | Matched MS Train",
            "YD | Primary | Matched MS Train",
        )
        add(
            f"YD | {collapsed_label} | Matched MS Train",
            f"YD | {source_split} | Matched MS Train",
        )

        add(
            "YD | Primary | Assay Validation",
            "YD | Primary | Assay Validation",
        )
        add(
            f"YD | {collapsed_label} | Assay Validation",
            f"YD | {source_split} | Assay Validation",
        )

        add(
            "YD | Primary | Matched MS Validation",
            "YD | Primary | Matched MS Validation",
        )
        add(
            f"YD | {collapsed_label} | Matched MS Validation",
            f"YD | {source_split} | Matched MS Validation",
        )

    add("YD | Test | Assay","YD | Test | Assay")
    add("YD | Test | Matched MS","YD | Test | Matched MS")

    # ============================================================
    # EPISCAN R9 TRAIN
    # ============================================================
    r9_train_suffix="Assay Train (R9 + RT positives)"

    if collapse_all_cv_splits:
        add(
            f"EpiScan R9 | {collapsed_label} | Assay Train",
            f"EpiScan R9 | {source_split} | {r9_train_suffix}",
        )
        add(
            f"EpiScan R9 | {collapsed_label} | Matched MS Train",
            f"EpiScan R9 | {source_split} | Matched MS Train",
        )

    else:
        add(
            "EpiScan R9 | Primary | Assay Train",
            "EpiScan R9 | Primary | Assay Train (R9 + RT positives)",
        )
        add(
            f"EpiScan R9 | {collapsed_label} | Assay Train",
            f"EpiScan R9 | {source_split} | Assay Train (R9 + RT positives)",
        )

        add(
            "EpiScan R9 | Primary | Matched MS Train",
            "EpiScan R9 | Primary | Matched MS Train",
        )
        add(
            f"EpiScan R9 | {collapsed_label} | Matched MS Train",
            f"EpiScan R9 | {source_split} | Matched MS Train",
        )

    # ============================================================
    # EPISCAN R9/RT TRAIN
    # ============================================================
    r9rt_train_suffix="Assay Train (R9 + RT all)"

    if collapse_all_cv_splits:
        add(
            f"EpiScan R9/RT | {collapsed_label} | Assay Train",
            f"EpiScan R9/RT | {source_split} | {r9rt_train_suffix}",
        )
        add(
            f"EpiScan R9/RT | {collapsed_label} | Matched MS Train",
            f"EpiScan R9/RT | {source_split} | Matched MS Train",
        )

    else:
        add(
            "EpiScan R9/RT | Primary | Assay Train",
            "EpiScan R9/RT | Primary | Assay Train (R9 + RT all)",
        )
        add(
            f"EpiScan R9/RT | {collapsed_label} | Assay Train",
            f"EpiScan R9/RT | {source_split} | Assay Train (R9 + RT all)",
        )

        add(
            "EpiScan R9/RT | Primary | Matched MS Train",
            "EpiScan R9/RT | Primary | Matched MS Train",
        )
        add(
            f"EpiScan R9/RT | {collapsed_label} | Matched MS Train",
            f"EpiScan R9/RT | {source_split} | Matched MS Train",
        )

    # ============================================================
    # EPISCAN VALIDATION
    # ============================================================
    if collapse_all_cv_splits:
        add(
            f"EpiScan R9 | {collapsed_label} | Assay Validation",
            f"EpiScan | {source_split} | R9 Validation | Assay",
        )
        add(
            f"EpiScan RT | {collapsed_label} | Assay Validation",
            f"EpiScan | {source_split} | RT Validation | Assay",
        )
        add(
            f"EpiScan R9/RT | {collapsed_label} | Matched MS Validation",
            f"EpiScan | {source_split} | Matched MS Validation (shared R9/RT)",
        )

    else:
        add(
            "EpiScan R9 | Primary | Assay Validation",
            "EpiScan | Primary | R9 Validation | Assay",
        )
        add(
            f"EpiScan R9 | {collapsed_label} | Assay Validation",
            f"EpiScan | {source_split} | R9 Validation | Assay",
        )

        add(
            "EpiScan RT | Primary | Assay Validation",
            "EpiScan | Primary | RT Validation | Assay",
        )
        add(
            f"EpiScan RT | {collapsed_label} | Assay Validation",
            f"EpiScan | {source_split} | RT Validation | Assay",
        )

        add(
            "EpiScan R9/RT | Primary | Matched MS Validation",
            "EpiScan | Primary | Matched MS Validation (shared R9/RT)",
        )
        add(
            f"EpiScan R9/RT | {collapsed_label} | Matched MS Validation",
            f"EpiScan | {source_split} | Matched MS Validation (shared R9/RT)",
        )

    # ============================================================
    # EPISCAN TEST
    # ============================================================
    add(
        "EpiScan R9 | Test | Assay",
        "EpiScan | R9 Test | Assay",
    )
    add(
        "EpiScan R9 | Test | Matched MS",
        "EpiScan | R9 Test | Matched MS",
    )

    add(
        "EpiScan RT | Test | Assay",
        "EpiScan | RT Test | Assay",
    )
    add(
        "EpiScan RT | Test | Matched MS",
        "EpiScan | RT Test | Matched MS",
    )

    # ============================================================
    # BA
    # ============================================================
    if collapse_all_cv_splits:
        add(
            f"BA | {collapsed_label} | Assay Train",
            source("BA","Assay Train"),
        )
        add(
            f"BA | {collapsed_label} | Matched MS Train",
            source("BA","Matched MS Train"),
        )
        add(
            f"BA | {collapsed_label} | Assay Validation",
            source("BA","Assay Validation"),
        )
        add(
            f"BA | {collapsed_label} | Matched MS Validation",
            source("BA","Matched MS Validation"),
        )

    else:
        add(
            "BA | Primary | Assay Train",
            "BA | Primary | Assay Train",
        )
        add(
            f"BA | {collapsed_label} | Assay Train",
            f"BA | {source_split} | Assay Train",
        )

        add(
            "BA | Primary | Matched MS Train",
            "BA | Primary | Matched MS Train",
        )
        add(
            f"BA | {collapsed_label} | Matched MS Train",
            f"BA | {source_split} | Matched MS Train",
        )

        add(
            "BA | Primary | Assay Validation",
            "BA | Primary | Assay Validation",
        )
        add(
            f"BA | {collapsed_label} | Assay Validation",
            f"BA | {source_split} | Assay Validation",
        )

        add(
            "BA | Primary | Matched MS Validation",
            "BA | Primary | Matched MS Validation",
        )
        add(
            f"BA | {collapsed_label} | Matched MS Validation",
            f"BA | {source_split} | Matched MS Validation",
        )

    add("BA | Test | Assay","BA | Test | Assay")
    add("BA | Test | Matched MS","BA | Test | Matched MS")

    # ============================================================
    # ESCAPE-SEQ
    #
    # If collapse_all_cv_splits=True, representative_split controls
    # which of the 5 splits supplies the displayed count table.
    # ============================================================
    if collapse_all_cv_splits:
        add(
            f"ESCAPE-seq | {collapsed_label} | Assay Train",
            source("ESCAPE-seq","Assay Train"),
        )
        add(
            f"ESCAPE-seq | {collapsed_label} | Matched MS Train",
            source("ESCAPE-seq","Matched MS Train"),
        )
        add(
            f"ESCAPE-seq | {collapsed_label} | Assay Validation",
            source("ESCAPE-seq","Assay Validation"),
        )
        add(
            f"ESCAPE-seq | {collapsed_label} | Matched MS Validation",
            source("ESCAPE-seq","Matched MS Validation"),
        )

    else:
        add(
            "ESCAPE-seq | Primary | Assay Train",
            "ESCAPE-seq | Primary | Assay Train",
        )
        add(
            f"ESCAPE-seq | {collapsed_label} | Assay Train",
            f"ESCAPE-seq | {source_split} | Assay Train",
        )

        add(
            "ESCAPE-seq | Primary | Matched MS Train",
            "ESCAPE-seq | Primary | Matched MS Train",
        )
        add(
            f"ESCAPE-seq | {collapsed_label} | Matched MS Train",
            f"ESCAPE-seq | {source_split} | Matched MS Train",
        )

        add(
            "ESCAPE-seq | Primary | Assay Validation",
            "ESCAPE-seq | Primary | Assay Validation",
        )
        add(
            f"ESCAPE-seq | {collapsed_label} | Assay Validation",
            f"ESCAPE-seq | {source_split} | Assay Validation",
        )

        add(
            "ESCAPE-seq | Primary | Matched MS Validation",
            "ESCAPE-seq | Primary | Matched MS Validation",
        )
        add(
            f"ESCAPE-seq | {collapsed_label} | Matched MS Validation",
            f"ESCAPE-seq | {source_split} | Matched MS Validation",
        )

    add(
        "ESCAPE-seq | Test | Assay",
        "ESCAPE-seq | Test | Assay",
    )
    add(
        "ESCAPE-seq | Test | Matched MS",
        "ESCAPE-seq | Test | Matched MS",
    )

    # ============================================================
    # REBUILD SUMMARY TABLE
    # ============================================================
    summary_rows=[]

    for name,df in collapsed_tables.items():
        n_pos=int(df["Positive peptide-alleles"].sum())
        n_neg=int(df["Negative peptide-alleles"].sum())
        n_total=int(df["Total peptide-alleles"].sum())

        if n_pos==0 and n_neg==0:
            ratio="NA"
        elif n_pos==0:
            ratio="0:1"
        elif n_neg==0:
            ratio="1:0"
        else:
            r=n_neg/n_pos
            ratio=(
                f"1:{int(round(r))}"
                if np.isclose(r,round(r))
                else f"1:{r:.2f}"
            )

        summary_rows.append({
            "Split":name,
            "Total alleles":int(df["Allele"].nunique()),
            "Positive peptide-alleles":n_pos,
            "Negative peptide-alleles":n_neg,
            "Total peptide-alleles":n_total,
            "Positive to negative ratio":ratio,
        })

    collapsed_summary=pd.DataFrame(summary_rows)

    if verbose:
        print("\n"+"="*90)
        print("COLLAPSED DATASET COUNT TABLES")
        print("="*90)
        print(f"Original tables: {len(dataset_count_tables):,}")
        print(f"Collapsed tables: {len(collapsed_tables):,}")

        if collapse_all_cv_splits:
            print("Collapse mode: Primary + CV1-CV4 -> CV splits (n=5)")
            print(f"Representative count table: {representative_split}")
        else:
            print("Collapse mode: Primary separate; CV1-CV4 -> Additional CV splits (n=4)")
            print(f"Representative additional CV table: {source_split}")

    return collapsed_tables,collapsed_summary

def insert_full_assay_datasets(
    count_tables,
    summary_df,
    ba_cls_full,
    escape_cls_full,
    escape_con_full,
):
    """
    Insert BA CLS Full, ESCAPE-seq CLS Full, and ESCAPE-seq CON Full
    immediately after Full MS Test in both the count-table dictionary
    and summary table.

    Does not modify the input dictionary/dataframe in place.
    """
    new_datasets={
        "BA CLS Full":ba_cls_full,
        "ESCAPE-seq CLS Full":escape_cls_full,
        "ESCAPE-seq CON Full":escape_con_full,
    }

    # ------------------------------------------------------------
    # Create per-allele count tables using the same counting helper.
    # ------------------------------------------------------------
    new_count_tables={
        name:count_peptide_alleles_by_allele(df)
        for name,df in new_datasets.items()
    }

    # ------------------------------------------------------------
    # Insert immediately after Full MS Test.
    # ------------------------------------------------------------
    out_tables={}

    found_ms_test=False
    for key,value in count_tables.items():
        out_tables[key]=value.copy()

        if key=="Full MS Test":
            found_ms_test=True
            for new_key,new_table in new_count_tables.items():
                out_tables[new_key]=new_table

    if not found_ms_test:
        raise KeyError("'Full MS Test' not found in count_tables.")

    # ------------------------------------------------------------
    # Generate summary rows using exactly the same summary helper.
    # ------------------------------------------------------------
    new_summary=summarize_count_tables(new_count_tables)

    if "Split" not in summary_df.columns:
        raise KeyError("summary_df must contain a 'Split' column.")

    ms_idx=summary_df.index[
        summary_df["Split"].eq("Full MS Test")
    ]

    if len(ms_idx)!=1:
        raise ValueError(
            f"Expected exactly one 'Full MS Test' summary row, found {len(ms_idx)}."
        )

    pos=summary_df.index.get_loc(ms_idx[0])

    out_summary=pd.concat(
        [
            summary_df.iloc[:pos+1].copy(),
            new_summary,
            summary_df.iloc[pos+1:].copy(),
        ],
        ignore_index=True,
    )

    print("Inserted:")
    for name in new_datasets:
        x=new_count_tables[name]
        print(
            f"  {name}: "
            f"{x['Allele'].nunique():,} alleles | "
            f"{x['Total peptide-alleles'].sum():,} peptide-alleles"
        )

    return out_tables,out_summary

def append_multidataset_updating_counts(
    count_tables,
    summary_df,
    train_sets,
    test_sets,
    train_keys,
    test_keys,
):
    """
    Append multi-dataset updating train/test count tables and summary rows.

    Parameters
    ----------
    count_tables : dict[str, pd.DataFrame]
        Existing per-allele count-table dictionary.

    summary_df : pd.DataFrame
        Existing summary table with one row per count-table entry.

    train_sets : dict[str, pd.DataFrame]
        Dictionary containing the multi-dataset updating training sets.

    test_sets : dict[str, pd.DataFrame]
        Dictionary containing the multi-dataset updating test sets.

    train_keys : dict
        Mapping canonical dataset name -> key in train_sets.

        Required canonical keys:
            "yd"
            "episcan"
            "ba"
            "escape"

        Example:
            {
                "yd":"yd_train",
                "episcan":"episcan_r9rt_train",
                "ba":"ba_train",
                "escape":"escape_train",
            }

    test_keys : dict
        Same structure, but keys point into test_sets.

    Returns
    -------
    out_tables : dict[str, pd.DataFrame]
        Existing count tables with multi-dataset updating tables appended.

    out_summary : pd.DataFrame
        Existing summary with corresponding rows appended.
    """
    required={"yd","episcan","ba","escape"}

    missing_train=required-set(train_keys)
    missing_test=required-set(test_keys)

    if missing_train:
        raise KeyError(
            f"train_keys missing required canonical keys: {sorted(missing_train)}"
        )

    if missing_test:
        raise KeyError(
            f"test_keys missing required canonical keys: {sorted(missing_test)}"
        )

    display_names={
        "yd":"YD Multi-dataset Updating",
        "episcan":"EpiScan R9/RT Multi-dataset Updating",
        "ba":"BA Multi-dataset Updating",
        "escape":"ESCAPE-seq Multi-dataset Updating",
    }

    dataset_order=["yd","episcan","ba","escape"]

    # ------------------------------------------------------------
    # Validate selected source keys.
    # ------------------------------------------------------------
    for dataset in dataset_order:
        train_key=train_keys[dataset]
        test_key=test_keys[dataset]

        if train_key not in train_sets:
            raise KeyError(
                f"Train key {train_key!r} for {dataset!r} not found in train_sets."
            )

        if test_key not in test_sets:
            raise KeyError(
                f"Test key {test_key!r} for {dataset!r} not found in test_sets."
            )

    # ------------------------------------------------------------
    # Copy existing outputs.
    # ------------------------------------------------------------
    out_tables={
        key:value.copy()
        for key,value in count_tables.items()
    }

    new_tables={}

    # ============================================================
    # ALL TRAIN SETS FIRST
    # ============================================================
    for dataset in dataset_order:
        source_key=train_keys[dataset]
        output_key=f"{display_names[dataset]} | Train"

        if output_key in out_tables:
            raise KeyError(
                f"Output key already exists: {output_key}"
            )

        new_tables[output_key]=count_peptide_alleles_by_allele(
            train_sets[source_key]
        )

    # ============================================================
    # ALL TEST SETS AFTER
    # ============================================================
    for dataset in dataset_order:
        source_key=test_keys[dataset]
        output_key=f"{display_names[dataset]} | Test"

        if output_key in out_tables or output_key in new_tables:
            raise KeyError(
                f"Output key already exists: {output_key}"
            )

        new_tables[output_key]=count_peptide_alleles_by_allele(
            test_sets[source_key]
        )

    # Append to end of dict in requested order.
    for key,value in new_tables.items():
        out_tables[key]=value

    # ------------------------------------------------------------
    # Create corresponding summary rows using the same helper.
    # ------------------------------------------------------------
    new_summary=summarize_count_tables(
        new_tables
    )

    out_summary=pd.concat(
        [
            summary_df.copy(),
            new_summary,
        ],
        ignore_index=True,
    )

    print("Appended multi-dataset updating sets:")

    for key,table in new_tables.items():
        print(
            f"  {key}: "
            f"{table['Allele'].nunique():,} alleles | "
            f"{table['Positive peptide-alleles'].sum():,} positive | "
            f"{table['Negative peptide-alleles'].sum():,} negative | "
            f"{table['Total peptide-alleles'].sum():,} total"
        )

    return out_tables,out_summary

def append_iedb_tcell_counts(
    count_tables,
    summary_df,
    recognition_sets,
    activation_sets,
    label_col="label",
):
    """
    Append IEDB T cell Recognition and Activation assay count tables.

    Expected dictionary keys
    ------------------------
    recognition_sets / activation_sets:
        "yd"
        "episcan_r9"
        "episcan_r9rt"
        "ba_con"
        "escape_con"

    Behavior
    --------
    - Recognition sets are appended first.
    - Activation sets are appended second.
    - These are treated as test/evaluation sets only.
    - If `label_col` exists, normal positive/negative counting is used.
    - If `label_col` does not exist, all rows are treated as positive.

    Returns
    -------
    out_tables : dict[str, pd.DataFrame]
        Existing count tables with IEDB tables appended.

    out_summary : pd.DataFrame
        Existing summary table with corresponding rows appended.
    """
    key_order=[
        "yd",
        "episcan_r9",
        "episcan_r9rt",
        "ba_con",
        "escape_con",
    ]

    display_names={
        "yd":"YD",
        "episcan_r9":"EpiScan R9",
        "episcan_r9rt":"EpiScan R9/RT",
        "ba_con":"BA",
        "escape_con":"ESCAPE-seq",
    }

    missing_rec=[k for k in key_order if k not in recognition_sets]
    missing_act=[k for k in key_order if k not in activation_sets]

    if missing_rec:
        raise KeyError(
            f"recognition_sets missing required keys: {missing_rec}"
        )

    if missing_act:
        raise KeyError(
            f"activation_sets missing required keys: {missing_act}"
        )

    def _count_iedb(df):
        df=df.copy()

        # If label is absent, these IEDB sets are positive-only.
        if label_col not in df.columns:
            df[label_col]=1

        return count_peptide_alleles_by_allele(
            df,
            allele_col="allele",
            label_col=label_col,
        )

    out_tables={
        k:v.copy()
        for k,v in count_tables.items()
    }

    new_tables={}

    # ============================================================
    # RECOGNITION FIRST
    # ============================================================
    for key in key_order:
        output_key=(
            f"{display_names[key]} | "
            "T cell Recognition Assays (IEDB)"
        )

        if output_key in out_tables:
            raise KeyError(
                f"Output key already exists: {output_key}"
            )

        new_tables[output_key]=_count_iedb(
            recognition_sets[key]
        )

    # ============================================================
    # ACTIVATION SECOND
    # ============================================================
    for key in key_order:
        output_key=(
            f"{display_names[key]} | "
            "T cell Activation Assays (IEDB)"
        )

        if output_key in out_tables or output_key in new_tables:
            raise KeyError(
                f"Output key already exists: {output_key}"
            )

        new_tables[output_key]=_count_iedb(
            activation_sets[key]
        )

    # Append all new tables in order.
    for key,value in new_tables.items():
        out_tables[key]=value

    # Build matching summary rows.
    new_summary=summarize_count_tables(
        new_tables
    )

    out_summary=pd.concat(
        [
            summary_df.copy(),
            new_summary,
        ],
        ignore_index=True,
    )

    print("Appended IEDB T cell assay sets:")

    for key,table in new_tables.items():
        print(
            f"  {key}: "
            f"{table['Allele'].nunique():,} alleles | "
            f"{table['Positive peptide-alleles'].sum():,} positive | "
            f"{table['Negative peptide-alleles'].sum():,} negative | "
            f"{table['Total peptide-alleles'].sum():,} total"
        )

    return out_tables,out_summary

import pandas as pd


def append_special_evaluation_set_counts(
    count_tables,
    summary_df,

    # EpiScan SARS-CoV-2
    episcan_cov_matched,
    episcan_cov_matched_ms,
    episcan_cov_transfer,
    episcan_cov_transfer_ms,

    # Multi-allelic MS pathogen sets
    ms_cov_ma,
    ms_zika_ma,
    ms_mtb_ma,

    # Positive-only evaluation sets
    escape_reported_neoantigens,
    ligand_mhc_atlas,

    allele_col="allele",
    label_col="label",
    verbose=True,
):
    """
    Append or overwrite specialized evaluation-set count tables and summary rows.

    Counting rules
    --------------
    1. EpiScan SARS-CoV-2 sets:
       Standard positive/negative counting from `label`.

    2. MS CoV/Zika/MTB MA sets:
       Multi-allelic MS sets. Report:
           Positive peptide-alleles = "-"
           Negative peptide-alleles = "-"
           Total peptide-alleles = row count
           Positive to negative ratio = "-"

    3. ESCAPE-seq reported neoantigens and Ligand.MHC Atlas:
       Positive-only sets regardless of whether a label column exists.
       Every row is counted as positive.

    Overwrite behavior
    ------------------
    If any of these named sets already exist in count_tables or summary_df,
    they are replaced rather than duplicated.

    Returns
    -------
    out_tables : dict[str, pd.DataFrame]
        Updated count-table dictionary.

    out_summary : pd.DataFrame
        Updated summary table with corresponding rows replaced/appended.
    """

    # ============================================================
    # HELPERS
    # ============================================================

    def _count_standard(df):
        return count_peptide_alleles_by_allele(
            df.copy(),
            allele_col=allele_col,
            label_col=label_col,
        )

    def _count_positive_only(df):
        x=df.copy()

        # Force every observation to positive regardless of any existing label.
        x[label_col]=1

        return count_peptide_alleles_by_allele(
            x,
            allele_col=allele_col,
            label_col=label_col,
        )

    def _count_total_only(df):
        """
        Per-allele counts for multi-allelic MS sets where positive/negative
        interpretation is intentionally not reported.
        """
        if allele_col not in df.columns:
            raise KeyError(
                f"Multi-allelic MS dataframe missing allele column "
                f"{allele_col!r}."
            )

        rows=[]

        for allele,g in df.groupby(
            allele_col,
            sort=True,
        ):
            rows.append({
                "Allele":str(allele),
                "Positive peptide-alleles":"-",
                "Negative peptide-alleles":"-",
                "Total peptide-alleles":int(len(g)),
                "Positive to negative ratio":"-",
            })

        return pd.DataFrame(
            rows,
            columns=[
                "Allele",
                "Positive peptide-alleles",
                "Negative peptide-alleles",
                "Total peptide-alleles",
                "Positive to negative ratio",
            ],
        )

    # ============================================================
    # DEFINE SETS IN EXACT REQUESTED ORDER
    # ============================================================

    set_specs=[
        (
            "EpiScan SARS-CoV-2 Matched Allele-Length Test Set",
            episcan_cov_matched,
            "standard",
        ),
        (
            "EpiScan SARS-CoV-2 Matched Allele-Length Test Set | Matched MS",
            episcan_cov_matched_ms,
            "standard",
        ),
        (
            "EpiScan SARS-CoV-2 Allele Transfer Learning Test Set",
            episcan_cov_transfer,
            "standard",
        ),
        (
            "EpiScan SARS-CoV-2 Allele Transfer Learning Test Set | Matched MS",
            episcan_cov_transfer_ms,
            "standard",
        ),
        (
            "MS CoV MA Test Set",
            ms_cov_ma,
            "total_only",
        ),
        (
            "MS Zika MA Test Set",
            ms_zika_ma,
            "total_only",
        ),
        (
            "MS MTB MA Test Set",
            ms_mtb_ma,
            "total_only",
        ),
        (
            "ESCAPE-seq Reported Neoantigens Test Set",
            escape_reported_neoantigens,
            "positive_only",
        ),
        (
            "MS Ligand.MHC Atlas Test Set",
            ligand_mhc_atlas,
            "positive_only",
        ),
    ]

    # ============================================================
    # COPY EXISTING TABLES
    # ============================================================

    out_tables={
        key:value.copy()
        for key,value in count_tables.items()
    }

    new_tables={}
    count_modes={}

    # ============================================================
    # BUILD NEW/UPDATED PER-ALLELE COUNT TABLES
    # ============================================================

    for name,df,mode in set_specs:

        if mode=="standard":
            table=_count_standard(df)

        elif mode=="positive_only":
            table=_count_positive_only(df)

        elif mode=="total_only":
            table=_count_total_only(df)

        else:
            raise ValueError(
                f"Unknown counting mode: {mode}"
            )

        new_tables[name]=table
        count_modes[name]=mode

    # ============================================================
    # OVERWRITE OR APPEND DICTIONARY ENTRIES
    # ============================================================

    for name,table in new_tables.items():
        out_tables[name]=table

    # ============================================================
    # BUILD NEW SUMMARY ROWS
    # ============================================================

    summary_rows=[]

    for name,table in new_tables.items():
        mode=count_modes[name]

        if mode=="total_only":
            summary_rows.append({
                "Split":name,
                "Total alleles":int(
                    table["Allele"].nunique()
                ),
                "Positive peptide-alleles":"-",
                "Negative peptide-alleles":"-",
                "Total peptide-alleles":int(
                    table["Total peptide-alleles"].sum()
                ),
                "Positive to negative ratio":"-",
            })

        else:
            n_pos=int(
                table["Positive peptide-alleles"].sum()
            )

            n_neg=int(
                table["Negative peptide-alleles"].sum()
            )

            n_total=int(
                table["Total peptide-alleles"].sum()
            )

            if n_pos==0 and n_neg==0:
                ratio="NA"

            elif n_pos==0:
                ratio="0:1"

            elif n_neg==0:
                ratio="1:0"

            else:
                r=n_neg/n_pos

                ratio=(
                    f"1:{int(round(r))}"
                    if abs(r-round(r))<1e-12
                    else f"1:{r:.2f}"
                )

            summary_rows.append({
                "Split":name,
                "Total alleles":int(
                    table["Allele"].nunique()
                ),
                "Positive peptide-alleles":n_pos,
                "Negative peptide-alleles":n_neg,
                "Total peptide-alleles":n_total,
                "Positive to negative ratio":ratio,
            })

    new_summary=pd.DataFrame(
        summary_rows
    )

    # ============================================================
    # REMOVE OLD SUMMARY ROWS, THEN APPEND UPDATED ONES
    # ============================================================

    replacement_names=set(
        new_tables.keys()
    )

    if "Split" not in summary_df.columns:
        raise KeyError(
            "summary_df must contain a 'Split' column."
        )

    out_summary=summary_df[
        ~summary_df["Split"].isin(
            replacement_names
        )
    ].copy()

    out_summary=pd.concat(
        [
            out_summary,
            new_summary,
        ],
        ignore_index=True,
    )

    # ============================================================
    # REPORT
    # ============================================================

    if verbose:
        print(
            "Added/updated specialized evaluation sets:"
        )

        for name,table in new_tables.items():
            mode=count_modes[name]

            if mode=="total_only":
                print(
                    f"  {name}: "
                    f"{table['Allele'].nunique():,} alleles | "
                    f"{table['Total peptide-alleles'].sum():,} total | "
                    "positive/negative = not reported"
                )

            else:
                print(
                    f"  {name}: "
                    f"{table['Allele'].nunique():,} alleles | "
                    f"{table['Positive peptide-alleles'].sum():,} positive | "
                    f"{table['Negative peptide-alleles'].sum():,} negative | "
                    f"{table['Total peptide-alleles'].sum():,} total"
                )

    return out_tables,out_summary

def rename_cv_count_table_keys(
    count_tables,
    summary_df,
    verbose=True,
):
    """
    Rename collapsed 5-CV dataset count-table keys to compact Excel-friendly
    names while preserving dictionary and summary-table order.

    Examples
    --------
    YD | CV splits (n=5) | Assay Train
        -> YD | Assay Train | 5 CV

    YD | CV splits (n=5) | Matched MS Train
        -> YD | MS Train | 5 CV

    YD | CV splits (n=5) | Assay Validation
        -> YD | Assay Val | 5 CV

    YD | CV splits (n=5) | Matched MS Validation
        -> YD | MS Val | 5 CV

    Non-CV keys are unchanged.
    """

    marker=" | CV splits (n=5) | "

    suffix_map={
        "Assay Train":"Assay Train",
        "Matched MS Train":"MS Train",
        "Assay Validation":"Assay Val",
        "Matched MS Validation":"MS Val",
    }

    def _rename(name):
        name=str(name)

        if marker not in name:
            return name

        dataset,suffix=name.split(marker,1)

        if suffix not in suffix_map:
            raise ValueError(
                f"Unrecognized 5-CV suffix in key:\n{name}"
            )

        return f"{dataset} | {suffix_map[suffix]} | 5 CV"

    # ------------------------------------------------------------
    # Rename dictionary keys while preserving insertion order.
    # ------------------------------------------------------------
    out_tables={}

    for old_key,df in count_tables.items():
        new_key=_rename(old_key)

        if new_key in out_tables:
            raise KeyError(
                f"Renaming creates duplicate key:\n"
                f"  old: {old_key}\n"
                f"  new: {new_key}"
            )

        out_tables[new_key]=df.copy()

        if verbose and old_key!=new_key:
            print(f"{old_key}")
            print(f"  -> {new_key}")

    # ------------------------------------------------------------
    # Rename corresponding summary rows.
    # ------------------------------------------------------------
    if "Split" not in summary_df.columns:
        raise KeyError("summary_df must contain a 'Split' column.")

    out_summary=summary_df.copy()
    out_summary["Split"]=out_summary["Split"].map(_rename)

    # Safety check.
    duplicates=out_summary["Split"].duplicated(keep=False)
    if duplicates.any():
        dup_names=out_summary.loc[duplicates,"Split"].tolist()
        raise ValueError(
            f"Renaming produced duplicate summary Split values: {dup_names}"
        )

    if verbose:
        n_changed=sum(
            old!=new
            for old,new in zip(
                count_tables.keys(),
                out_tables.keys(),
            )
        )
        print(f"\nRenamed {n_changed} CV count-table keys.")

    return out_tables,out_summary

def replace_count_table_keys(
    count_tables,
    summary_df,
    replacement_dict,
    split_col="Split",
    max_excel_length=31,
    verbose=True,
):
    """
    Replace selected count-table dictionary keys and corresponding summary
    Split values using the same explicit replacement dictionary.

    Dictionary insertion order and summary row order are preserved.
    Inputs are not modified in place.
    """
    if split_col not in summary_df.columns:
        raise KeyError(
            f"summary_df must contain column {split_col!r}."
        )

    # ------------------------------------------------------------
    # Validate replacement targets.
    # ------------------------------------------------------------
    missing=[
        old for old in replacement_dict
        if old not in count_tables
    ]

    if missing:
        raise KeyError(
            "Replacement keys not found in count_tables:\n"
            +"\n".join(f"  {x}" for x in missing)
        )

    too_long={
        old:new
        for old,new in replacement_dict.items()
        if len(new)>max_excel_length
    }

    if too_long:
        raise ValueError(
            "Some replacement names still exceed Excel's "
            f"{max_excel_length}-character limit:\n"
            +"\n".join(
                f"  {len(new):2d} | {new}"
                for new in too_long.values()
            )
        )

    # ------------------------------------------------------------
    # Rename dictionary keys while preserving order.
    # ------------------------------------------------------------
    out_tables={}

    for old_key,df in count_tables.items():
        new_key=replacement_dict.get(
            old_key,
            old_key,
        )

        if new_key in out_tables:
            raise KeyError(
                f"Key replacement creates duplicate key: {new_key!r}"
            )

        out_tables[new_key]=df.copy()

        if verbose and old_key!=new_key:
            print(
                f"{old_key}\n"
                f"  -> {new_key}"
            )

    # ------------------------------------------------------------
    # Rename corresponding summary rows.
    # ------------------------------------------------------------
    out_summary=summary_df.copy()

    out_summary[split_col]=out_summary[
        split_col
    ].replace(replacement_dict)

    # ------------------------------------------------------------
    # Validate final outputs.
    # ------------------------------------------------------------
    dup_summary=out_summary[
        split_col
    ].duplicated(keep=False)

    if dup_summary.any():
        duplicates=out_summary.loc[
            dup_summary,
            split_col,
        ].tolist()

        raise ValueError(
            f"Replacement creates duplicate summary names: {duplicates}"
        )

    still_too_long=[
        key
        for key in out_tables
        if len(str(key))>max_excel_length
    ]

    if still_too_long:
        print(
            "\nWARNING: These dictionary keys still exceed "
            f"{max_excel_length} characters:"
        )

        for key in still_too_long:
            print(
                f"  {len(key):2d} | {key}"
            )

    if verbose:
        print(
            f"\nReplaced {len(replacement_dict)} names."
        )

    return out_tables,out_summary

In [ ]:
%%time
dataset_count_tables,dataset_count_summary=build_primary_dataset_count_tables(
    DATA_DIR,
    verbose=True,
)

Loading full MS datasets once...
Loading CV split dictionaries...
Loading primary assay splits...

Adding full MS splits...
  Full MS Train: 35,913,223 rows | 192 alleles
  Full MS Validation: 2,075,300 rows | 142 alleles
  Full MS Test: 3,214,300 rows | 142 alleles

Adding fixed test splits...
  YD | Test | Assay: 160,000 rows | 16 alleles
  YD | Test | Matched MS: 158,400 rows | 16 alleles
  EpiScan | R9 Test | Assay: 40,000 rows | 4 alleles
  EpiScan | R9 Test | Matched MS: 40,000 rows | 4 alleles
  EpiScan | RT Test | Assay: 40,000 rows | 4 alleles
  EpiScan | RT Test | Matched MS: 40,000 rows | 4 alleles
  BA | Test | Assay: 4,221 rows | 51 alleles
  BA | Test | Matched MS: 1,361,400 rows | 51 alleles
  ESCAPE-seq | Test | Assay: 1,446 rows | 41 alleles
  ESCAPE-seq | Test | Matched MS: 361,900 rows | 37 alleles


Building assay train/validation counts:   0%|          | 0/20 [00:00<?, ?it/s]

  YD | Primary | Assay Train: 1,319,748 rows | 16 alleles


Building assay train/validation counts:   5%|▌         | 1/20 [00:15<04:47, 15.14s/it]

  YD | Primary | Matched MS Train: 4,912,237 rows | 16 alleles
  YD | Primary | Assay Validation: 72,300 rows | 16 alleles
  YD | Primary | Matched MS Validation: 80,000 rows | 16 alleles
  EpiScan R9 | Primary | Assay Train (R9 + RT positives): 1,970,293 rows | 4 alleles
  EpiScan R9 | Primary | Matched MS Train: 2,938,459 rows | 4 alleles
  EpiScan R9/RT | Primary | Assay Train (R9 + RT all): 2,498,576 rows | 4 alleles
  EpiScan R9/RT | Primary | Matched MS Train: 2,938,459 rows | 4 alleles


...

Building assay train/validation counts:  95%|█████████▌| 19/20 [09:42<00:29, 29.10s/it]

  BA | CV4 | Matched MS Train: 29,901,282 rows | 83 alleles
  BA | CV4 | Assay Validation: 3,052 rows | 41 alleles
  BA | CV4 | Matched MS Validation: 327,500 rows | 41 alleles
  ESCAPE-seq | CV4 | Assay Train: 69,313 rows | 49 alleles


Building assay train/validation counts: 100%|██████████| 20/20 [10:44<00:00, 32.22s/it]

  ESCAPE-seq | CV4 | Matched MS Train: 12,048,050 rows | 44 alleles
  ESCAPE-seq | CV4 | Assay Validation: 1,320 rows | 41 alleles
  ESCAPE-seq | CV4 | Matched MS Validation: 185,000 rows | 37 alleles



Complete: 108 per-allele count tables generated.
Summary table: 108 split-level rows.
CPU times: user 10min 4s, sys: 1min 40s, total: 11min 44s
Wall time: 11min 44s


In [ ]:
%%time
cv_count_report,cv_count_details=report_cv_count_consistency(
    dataset_count_tables,
    tolerance=2,
    cv_splits=(1,2,3,4),
    print_per_allele_differences=False,
)


CV SPLIT COUNT CONSISTENCY REPORT
Tolerance: ±2 peptide-allele rows per allele/count category

----------------------------------------------------------------------------------------------------
YD | Assay Train
----------------------------------------------------------------------------------------------------
Status: EXACT | Can collapse: True
Exact CV clusters: CV1,CV2,CV3,CV4
Alleles: 16
Positive peptide-alleles: 26,160
Negative peptide-alleles: 1,293,588
Total peptide-alleles: 1,319,748
Same allele membership: True
Maximum per-allele count difference: 0

----------------------------------------------------------------------------------------------------
YD | Matched MS Train
----------------------------------------------------------------------------------------------------
Status: EXACT | Can collapse: True
Exact CV clusters: CV1,CV2,CV3,CV4
Alleles: 16
Positive peptide-alleles: 97,087
Negative peptide-alleles: 4,815,150
Total peptide-alleles: 4,912,237
Same allele membership: 

In [ ]:
# collapse for clarity since most cv splits are identical counts
collapsed_dataset_count_tables,collapsed_dataset_count_summary=(
    collapse_dataset_count_tables(
        dataset_count_tables,
        collapse_all_cv_splits=True,
        representative_split="Primary",
        verbose=True,
    )
)

Full MS Train
  <- Full MS Train
Full MS Validation
  <- Full MS Validation
Full MS Test
  <- Full MS Test
YD | CV splits (n=5) | Assay Train
  <- YD | Primary | Assay Train
YD | CV splits (n=5) | Matched MS Train
  <- YD | Primary | Matched MS Train
YD | CV splits (n=5) | Assay Validation
  <- YD | Primary | Assay Validation
YD | CV splits (n=5) | Matched MS Validation
  <- YD | Primary | Matched MS Validation
YD | Test | Assay
  <- YD | Test | Assay
YD | Test | Matched MS
  <- YD | Test | Matched MS
EpiScan R9 | CV splits (n=5) | Assay Train
  <- EpiScan R9 | Primary | Assay Train (R9 + RT positives)
EpiScan R9 | CV splits (n=5) | Matched MS Train
  <- EpiScan R9 | Primary | Matched MS Train
EpiScan R9/RT | CV splits (n=5) | Assay Train
  <- EpiScan R9/RT | Primary | Assay Train (R9 + RT all)
EpiScan R9/RT | CV splits (n=5) | Matched MS Train
  <- EpiScan R9/RT | Primary | Matched MS Train
EpiScan R9 | CV splits (n=5) | Assay Validation
  <- EpiScan | Primary | R9 Validation | Assay


In [ ]:
# next add the full BA CLS and ESCAPE-seq CLS and CON

# ba test sets
ba_full_cls_test_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_cls_test_set.csv", index_col=0)

# escape seq test set
escape_seq_datasets_dir = DATA_DIR / "Training_Datasets/Processed/ESCAPE_seq"
escape_seq_full_cls_test_set=pd.read_csv(f"{escape_seq_datasets_dir}/escape_seq_full_cls_test_set.csv", index_col=0)
escape_seq_full_con_test_set=pd.read_csv(f"{escape_seq_datasets_dir}/escape_seq_full_con_test_set.csv", index_col=0)

In [ ]:
# insert for model benchmarking
collapsed_dataset_count_tables,collapsed_dataset_count_summary=(
    insert_full_assay_datasets(
        collapsed_dataset_count_tables,
        collapsed_dataset_count_summary,
        ba_cls_full=ba_full_cls_test_set,
        escape_cls_full=escape_seq_full_cls_test_set,
        escape_con_full=escape_seq_full_con_test_set,
    )
)

Inserted:
  BA CLS Full: 71 alleles | 111,500 peptide-alleles
  ESCAPE-seq CLS Full: 44 alleles | 55,900 peptide-alleles
  ESCAPE-seq CON Full: 44 alleles | 14,456 peptide-alleles


In [ ]:
#### Load and process datasets for multi dataset updating

# MS: use the ablation training set + standard validation/test sets.
ms_datasets_dir=DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{ms_datasets_dir}/ms_train_data_remaining_minimal.csv",index_col=0)
ms_val_data=pd.read_csv(f"{ms_datasets_dir}/ms_val_data.csv",index_col=0)
ms_test_data=pd.read_csv(f"{ms_datasets_dir}/ms_test_data.csv",index_col=0)

# YD / Holec: use the ablation train-remaining set + validation/test sets.
holec_datasets_dir=DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
holec_train_set=pd.read_csv(f"{holec_datasets_dir}/holec_train_set_remaining.csv",index_col=0)
holec_val_set=pd.read_csv(f"{holec_datasets_dir}/holec_val_set.csv",index_col=0)
holec_test_set=pd.read_csv(f"{holec_datasets_dir}/holec_test_set.csv",index_col=0)

# EpiScan R9/RT: use both train-remaining sets and both validation/test sets.
episcan_train_test_dir=DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
episcan_r9_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_train_set_remaining.csv",index_col=0)
episcan_rt_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_train_set_remaining.csv",index_col=0)
episcan_r9_val_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_val_set.csv",index_col=0)
episcan_rt_val_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_val_set.csv",index_col=0)
episcan_r9_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv",index_col=0)
episcan_rt_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv",index_col=0)
sd3_formatted=pd.read_csv(f"{episcan_train_test_dir}/sd3_formatted.csv",index_col=0)

# Preserve source before merging EpiScan R9/RT validation/test sets.
episcan_r9_val_set["dataset_source"]="r9"
episcan_rt_val_set["dataset_source"]="rt"
episcan_r9_test_set["dataset_source"]="r9"
episcan_rt_test_set["dataset_source"]="rt"

# BA: pre-created multi-dataset classification train/test split.
ba_train_test_dir=DATA_DIR / "Training_Datasets" / "Processed" / "BA"
ba_train_set=pd.read_csv(f"{ba_train_test_dir}/ba_train_set_for_md_updating.csv",index_col=0)
ba_test_set=pd.read_csv(f"{ba_train_test_dir}/ba_test_set_for_md_updating.csv",index_col=0)

# ESCAPE-seq: pre-created multi-dataset classification train/test split.
escape_train_test_dir=DATA_DIR / "Training_Datasets" / "Processed" / "ESCAPE_seq"
escape_train_set=pd.read_csv(f"{escape_train_test_dir}/escape_train_set_for_md_updating.csv",index_col=0)
escape_test_set=pd.read_csv(f"{escape_train_test_dir}/escape_test_set_for_md_updating.csv",index_col=0)

print("Processing train, validation, and test sets...")
def processing_wrapper(df):
    df=full_process_data(df,mhc_seq_dict,remove_flank=True,add_pseudo_label=False)
    df=create_seq_allele_column(df)
    df["peptide"]=df.seq.str.replace("$","")
    df["seq_len"]=df.seq.str.replace("$","").apply(len)
    return df

ms_train_data=processing_wrapper(ms_train_data)
ms_val_data=processing_wrapper(ms_val_data)
ms_test_data=processing_wrapper(ms_test_data)

holec_train_set=processing_wrapper(holec_train_set)
holec_val_set=processing_wrapper(holec_val_set)
holec_test_set=processing_wrapper(holec_test_set)

episcan_r9_train_set=processing_wrapper(episcan_r9_train_set)
episcan_rt_train_set=processing_wrapper(episcan_rt_train_set)
episcan_r9_val_set=processing_wrapper(episcan_r9_val_set)
episcan_rt_val_set=processing_wrapper(episcan_rt_val_set)
episcan_r9_test_set=processing_wrapper(episcan_r9_test_set)
episcan_rt_test_set=processing_wrapper(episcan_rt_test_set)
sd3_formatted=processing_wrapper(sd3_formatted)

ba_train_set=processing_wrapper(ba_train_set)
ba_test_set=processing_wrapper(ba_test_set)

escape_train_set=processing_wrapper(escape_train_set)
escape_test_set=processing_wrapper(escape_test_set)
print(end="\n")

# merge episcan sets
print("Merging EpiScan R9/RT sets...")
episcan_train_set=pd.concat([episcan_r9_train_set,episcan_rt_train_set])
episcan_val_set=pd.concat([episcan_r9_val_set,episcan_rt_val_set])
episcan_test_set=pd.concat([episcan_r9_test_set,episcan_rt_test_set])
print_data_stats(episcan_train_set)
print_data_stats(episcan_val_set)
print_data_stats(episcan_test_set)
print(end="\n")

# aggregate sets

# Replay set is held separately to create the adaptive buffer.
train_sets={
    "ms":ms_train_data,
    "yd":holec_train_set,
    "episcan":episcan_train_set,
    "ba":ba_train_set,
    "escape":escape_train_set,
}

# BA and ESCAPE-seq intentionally have no validation sets for this classification-updating experiment.
test_sets={
    "ms_test":ms_test_data,
    "yd_test":holec_test_set,
    "episcan_test":episcan_test_set,
    "ba_test":ba_test_set,
    "escape_test":escape_test_set,
    "cov_test":sd3_formatted,
}

val_sets={
    "ms_val":ms_val_data,
    "yd_val":holec_val_set,
    "episcan_val":episcan_val_set,
}
# filter, later we can keep escore and ba_value
filtered_columns=['seq','allele','label','seq_allele','n_flank','c_flank','mhc_a_1','mhc_b_1','mhc_c_1','mhc_a_2','mhc_b_2','mhc_c_2','sa_ma','mhcprime','dataset_source', 'escore', 'ba_value']
test_sets=filter_dict_columns(test_sets,filtered_columns)
val_sets=filter_dict_columns(val_sets,filtered_columns)

# rekey
test_sets = {(k[:-5] if k.endswith("_test") else k): v for k, v in test_sets.items()}
val_sets = {(k.removesuffix("_val") if k.endswith("_val") else k): v for k, v in val_sets.items()}

print("Cleaning train/test cross-dataset peptide-allele overlaps...")
seq_allele_overlap_matrix,train_sets=seq_allele_overlap_table(train_sets,test_sets,seq_col="seq",allele_col="allele",remove_test_from_train=True)
print(end="\n")

Processing train, validation, and test sets...

Merging EpiScan R9/RT sets...
Total: 2,498,576
Pos: 108,158 | 4.33%
Neg: 2,390,418 | 95.67%
N alleles: 4
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4
Total: 80,000
Pos: 800 | 1.00%
Neg: 79,200 | 99.00%
N alleles: 4

Cleaning train/test cross-dataset peptide-allele overlaps...


Filtering train sets: 100%|██████████| 5/5 [00:48<00:00,  9.63s/it]

In [ ]:
# add multi-dataset updating stuff
train_keys={
    "yd":"yd",
    "episcan":"episcan",
    "ba":"ba",
    "escape":"escape",
}

test_keys={
    "yd":"yd",
    "episcan":"episcan",
    "ba":"ba",
    "escape":"escape",
}

collapsed_dataset_count_tables,collapsed_dataset_count_summary=(
    append_multidataset_updating_counts(
        count_tables=collapsed_dataset_count_tables,
        summary_df=collapsed_dataset_count_summary,
        train_sets=train_sets,
        test_sets=test_sets,
        train_keys=train_keys,
        test_keys=test_keys,
    )
)

Appended multi-dataset updating sets:
  YD Multi-dataset Updating | Train: 16 alleles | 26,011 positive | 1,286,318 negative | 1,312,329 total
  EpiScan R9/RT Multi-dataset Updating | Train: 4 alleles | 108,076 positive | 2,388,233 negative | 2,496,309 total
  BA Multi-dataset Updating | Train: 83 alleles | 38,516 positive | 69,621 negative | 108,137 total
  ESCAPE-seq Multi-dataset Updating | Train: 44 alleles | 6,807 positive | 16,928 negative | 23,735 total
  YD Multi-dataset Updating | Test: 16 alleles | 1,600 positive | 158,400 negative | 160,000 total
  EpiScan R9/RT Multi-dataset Updating | Test: 4 alleles | 800 positive | 79,200 negative | 80,000 total
  BA Multi-dataset Updating | Test: 61 alleles | 610 positive | 44,603 negative | 45,213 total
  ESCAPE-seq Multi-dataset Updating | Test: 42 alleles | 420 positive | 41,447 negative | 41,867 total


In [ ]:
%%time
# immunogenicity sets

final_recognition_datasets={}
final_activation_datasets={}

def avg_score_per_allele(df, score_cols, allele_col="allele"):
    cols = [c for c in score_cols if c in df.columns]
    return df.groupby(allele_col, sort=False)[cols].mean()

def average_prefix_groups(df, prefix_map, suffix=None, collapse_alleles=False):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    nonnumeric_cols = [c for c in df.columns if c not in numeric_cols]

    if suffix is not None:
        eligible = [c for c in numeric_cols if c.endswith(suffix)]
    else:
        eligible = list(numeric_cols)

    matched = set()
    score_out = pd.DataFrame(index=df.index)
    for final_name, prefix in prefix_map.items():
        cols = [c for c in eligible if c.startswith(prefix)]
        if cols:
            score_out[final_name] = df[cols].mean(axis=1)
            matched.update(cols)
    for c in eligible:
        if c in matched:
            continue
        name = c[:-len(suffix)] if suffix is not None else c
        score_out[name] = df[c]

    if collapse_alleles:
        return score_out.mean(axis=0).to_frame("avg_score").rename_axis("score_col")

    out = pd.concat([df[nonnumeric_cols], score_out], axis=1)
    return out

def pct_recovered_per_allele(df, score_cols, threshold, allele_col="allele"):
    def frac(g):
        return pd.Series({col: (g[col] >= threshold).sum() / len(g) * 100 for col in score_cols})
    out = df.groupby(allele_col, sort=False).apply(frac)
    out.index.name = allele_col
    return out

# prefix helper
def build_score_cols(prefixes, seeds):
    return [f"{prefix}_seed_{seed}" for prefix in prefixes for seed in seeds]

def filter_alleles_min_samples(df, n, allele_col="allele"):
    counts = df[allele_col].value_counts()
    keep = counts.index[counts >= n]
    return df[df[allele_col].isin(keep)].copy()

def recovery_table_over_thresholds(df, score_cols, thresholds, prefix_map, allele_col="allele", base_col=None, per_allele=True):
    df = df.reset_index(drop=True)
    rec_cols = list(score_cols) if (base_col is None or base_col in score_cols) else list(score_cols) + [base_col]

    def _collapse(s):
        matched, out = set(), {}
        for final_name, prefix in prefix_map.items():
            cols = [c for c in s.index if c.startswith(prefix)]
            if cols:
                out[final_name] = s[cols].mean()
                matched.update(cols)
        for c in s.index:
            if c not in matched:
                out[c] = s[c]
        return pd.Series(out)

    rows = {}
    for t in thresholds:
        if per_allele:
            rec = (df[rec_cols] >= t).groupby(df[allele_col].values).mean() * 100 # rows=alleles, cols=score cols
            if base_col is not None:
                rec = rec.sub(rec[base_col], axis=0) # delta before seed-averaging
            s = rec.mean(axis=0) # average recovery across alleles
        else:
            s = (df[rec_cols] >= t).mean() * 100 # one "allele" = whole dataset
            if base_col is not None:
                s = s - s[base_col]
        rows[t] = _collapse(s) # prefix-average + keep non-prefix cols

    out = pd.DataFrame(rows).T
    out.index.name = "threshold"
    return out

def split_out_overlap(df1, df2, seq_col="seq", allele_col="allele"):
    df1 = df1.reset_index(drop=True); df2 = df2.reset_index(drop=True)
    key1 = list(zip(df1[seq_col], df1[allele_col]))
    key2 = set(zip(df2[seq_col], df2[allele_col]))
    mask1 = pd.Series([k in key2 for k in key1], index=df1.index)
    mask2 = pd.Series([k in set(key1) for k in zip(df2[seq_col], df2[allele_col])], index=df2.index)
    overlap_df = df1[mask1].reset_index(drop=True)
    print(f"Overlap: {overlap_df[[seq_col, allele_col]].drop_duplicates().shape[0]} seq_allele pairs")
    return df1[~mask1].reset_index(drop=True), df2[~mask2].reset_index(drop=True), overlap_df

# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data_remaining_minimal.csv", index_col=0)
ms_val_data=pd.read_csv(f"{datasets_dir}/ms_val_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

# process
ms_train_data=full_process_data(ms_train_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data=create_seq_allele_column(ms_train_data)

ms_val_data=full_process_data(ms_val_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_val_data=create_seq_allele_column(ms_val_data)

ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

#### YD train/val/test data

# yd train and test sets
holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
holec_train_set = pd.read_csv(f"{holec_datasets_dir}/holec_train_set_remaining.csv", index_col=0)
holec_val_set = pd.read_csv(f"{holec_datasets_dir}/holec_val_set.csv", index_col=0)
holec_test_set  = pd.read_csv(f"{holec_datasets_dir}/holec_test_set.csv", index_col=0)
# process
holec_train_set=full_process_data(holec_train_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_train_set=create_seq_allele_column(holec_train_set)

holec_val_set=full_process_data(holec_val_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_val_set=create_seq_allele_column(holec_val_set)

holec_test_set=full_process_data(holec_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_test_set=create_seq_allele_column(holec_test_set)

#### EpiScan R9 and R9/RT train/val/test data
# import
episcan_train_test_dir=DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
episcan_r9_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_train_set_remaining.csv",index_col=0)
episcan_rt_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_train_set_remaining.csv",index_col=0)
episcan_r9_val_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_val_set.csv",index_col=0)
episcan_rt_val_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_val_set.csv",index_col=0)
episcan_r9_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv",index_col=0)
episcan_rt_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv",index_col=0)
sd3_formatted=pd.read_csv(f"{episcan_train_test_dir}/sd3_formatted.csv",index_col=0)

episcan_r9_val_set["dataset_source"]="r9"
episcan_rt_val_set["dataset_source"]="rt"
episcan_r9_test_set["dataset_source"]="r9"
episcan_rt_test_set["dataset_source"]="rt"
# process
def processing_wrapper(df):
    df=full_process_data(df,mhc_seq_dict,remove_flank=True,add_pseudo_label=False)
    df=create_seq_allele_column(df)
    df["peptide"]=df.seq.str.replace("$","")
    df["seq_len"]=df.seq.str.replace("$","").apply(len)
    return df

episcan_r9_train_set=processing_wrapper(episcan_r9_train_set)
episcan_rt_train_set=processing_wrapper(episcan_rt_train_set)
episcan_r9_val_set=processing_wrapper(episcan_r9_val_set)
episcan_rt_val_set=processing_wrapper(episcan_rt_val_set)
episcan_r9_test_set=processing_wrapper(episcan_r9_test_set)
episcan_rt_test_set=processing_wrapper(episcan_rt_test_set)
sd3_formatted=processing_wrapper(sd3_formatted)

# merge episcan sets
print("Merging EpiScan R9/RT sets...")
episcan_train=pd.concat([episcan_r9_train_set,episcan_rt_train_set])
episcan_val=pd.concat([episcan_r9_val_set,episcan_rt_val_set])
episcan_test=pd.concat([episcan_r9_test_set,episcan_rt_test_set])
print_data_stats(episcan_train)
print_data_stats(episcan_val)
print_data_stats(episcan_test)
print(end="\n")

#### BA train/val/test data
# import
ba_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "BA"
ba_train=pd.read_csv(f"{ba_train_test_dir}/train_df_ba_for_updating_remaining.csv", index_col=0)
ba_val=pd.read_csv(f"{ba_train_test_dir}//ba_val_set.csv", index_col=0)
ba_test=pd.read_csv(f"{ba_train_test_dir}/test_df_ba_for_updating.csv", index_col=0)
# process
ba_train=full_process_data(ba_train, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_train=create_seq_allele_column(ba_train)

ba_val=full_process_data(ba_val, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_val=create_seq_allele_column(ba_val)

ba_test=full_process_data(ba_test, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ba_test=create_seq_allele_column(ba_test)

# no background, score file for escape (same as all other files)
#### ESCAPE-seq train/val/test data
# import 
escape_train=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_con_train_set_remaining_cleaned.csv", index_col=0)
escape_val=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_con_val_set_cleaned.csv", index_col=0)
escape_test=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_con_test_set_cleaned.csv", index_col=0)

# process
escape_train=full_process_data(escape_train, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_train=create_seq_allele_column(escape_train)

escape_val=full_process_data(escape_val, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_val=create_seq_allele_column(escape_val)

escape_test=full_process_data(escape_test, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escape_test=create_seq_allele_column(escape_test)

# load
iedb_t_cell_data_f_cleaned_holec=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_holec.csv", index_col=0)
iedb_t_cell_data_f_cleaned_episcan=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_episcan.csv", index_col=0)
iedb_t_cell_data_f_cleaned_ba=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_ba.csv", index_col=0)
iedb_t_cell_data_f_cleaned_escape=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/Immunogenicity_Data/iedb_t_cell_data_f_cleaned_escape.csv", index_col=0)

# fix col names
iedb_t_cell_data_f_cleaned_episcan.rename(columns={"mhcprime_abt_rapid": "mhcprime_abt", "mhcprime_abt_rapid_rank_r_chp": "mhcprime_abt_rank_r_chp"}, inplace=True)
iedb_t_cell_data_f_cleaned_ba.rename(columns={"mhcprime_abt_rapid": "mhcprime_abt", "mhcprime_abt_rapid_rank_r_chp": "mhcprime_abt_rank_r_chp"}, inplace=True)
iedb_t_cell_data_f_cleaned_escape.rename(columns={"mhcprime_abt_rapid": "mhcprime_abt", "mhcprime_abt_rapid_rank_r_chp": "mhcprime_abt_rank_r_chp"}, inplace=True)

# get score and rank score cols
dataset_prefix_dict={
    "yd": ["holec_ft_steps_300_lr_2eN4", "holec_pepcl_r_dr_steps_300_lr_2eN4"],
    "episcan_r9": ["episcan_r9_ft_steps_300_lr_2eN4", "episcan_r9_pepcl_r_dr_steps_300_lr_2eN4"],
    "episcan_r9rt": ["episcan_r9rt_ft_steps_400_lr_2eN4", "episcan_r9rt_pepcl_r_dr_steps_300_lr_2eN4"],
    "ba_con": ["ba_con_ft_steps_200_lr_2eN4", "ba_con_pepcl_r_dr_steps_300_lr_2eN4"],
    "escape_con": ["escape_con_ft_steps_200_lr_2eN4", "escape_con_pepcl_r_dr_steps_300_lr_2eN4"],
}

seeds=[0,1,2,3,4,5,6,7,8]

dataset_score_cols={}
dataset_rank_score_cols={}

for d, prefixes in dataset_prefix_dict.items():
    score_cols=["mhcprime_abt"]
    score_cols+=build_score_cols(prefixes, seeds)
    print(d, len(score_cols), score_cols)
    dataset_score_cols[d]=score_cols
    rank_score_cols=[f"{sc}_rank_r_chp" for sc in score_cols]
    # print(len(rank_score_cols), rank_score_cols)
    dataset_rank_score_cols[d]=rank_score_cols

# seq len helper
def add_seq_len(df):
    df["seq_len"]=df.seq.str.replace("$", "").apply(len)
    return df

# get per dataset train/val/test sets
train_val_test_per_dataset={
    "yd": {"train": holec_train_set, "val": holec_val_set, "test": holec_test_set},
    "episcan_r9": {"train": episcan_train, "val": episcan_val, "test": episcan_test},
    "episcan_r9rt": {"train": episcan_train, "val": episcan_val, "test": episcan_test}, # duplicate for simplicity
    "ba_con": {"train": ba_train, "val": ba_val, "test": ba_test},
    "escape_con": {"train": escape_train, "val": escape_val, "test": escape_test},
}

# get per dataset iedb data (dup episcan)
iedb_t_cell_data_f_cleaned_episcan_r9=iedb_t_cell_data_f_cleaned_episcan.copy()
iedb_t_cell_data_f_cleaned_episcan_r9rt=iedb_t_cell_data_f_cleaned_episcan.copy()

dataset_iedb_dfs={
    "yd": iedb_t_cell_data_f_cleaned_holec,
    "episcan_r9": iedb_t_cell_data_f_cleaned_episcan_r9,
    "episcan_r9rt": iedb_t_cell_data_f_cleaned_episcan_r9rt,
    "ba_con": iedb_t_cell_data_f_cleaned_ba,
    "escape_con": iedb_t_cell_data_f_cleaned_escape,
}

# datasets and method groups
datasets=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"]
recognition_methods=["multimer/tetramer", "binding assay", "SPR"] # prioritization of peptide-mhc-tcr recognized complexes
activation_methods=["ELISPOT", "ICS", "51Cr", "CFSE"] # for functional t cell responses

dataset_cleaned_and_grouped_iedb_data={}

for dataset in datasets:
    df=dataset_iedb_dfs[dataset].copy()
    df=add_seq_len(df)
    
    dataset_train=train_val_test_per_dataset[dataset]["train"]
    dataset_val=train_val_test_per_dataset[dataset]["val"]
    dataset_test=train_val_test_per_dataset[dataset]["test"]

    dataset_train=add_seq_len(dataset_train)
    dataset_val=add_seq_len(dataset_val)
    dataset_test=add_seq_len(dataset_test)

    # filter to matched and unmatched allele x length based on dataset
    if dataset in ["yd", "episcan_r9", "episcan_r9rt", "escape_con"]:
        df_matched_allele_length=df[df.allele.isin(dataset_train.allele.unique())].query("seq_len == 9")
    else:
        # for ba_con
        df_matched_allele_length, _=filter_by_allele_len_pairs(df, dataset_train, allele_col="allele", seq_col="seq", filter_both=False)

    # get pos only
    df_matched_allele_length=df_matched_allele_length.query("label == 1")

    # split activation and recognition methods
    df_matched_allele_length_recog=df_matched_allele_length[df_matched_allele_length.method.isin(recognition_methods)].drop_duplicates(subset=["seq_allele"]).copy()
    df_matched_allele_length_activation=df_matched_allele_length[df_matched_allele_length.method.isin(activation_methods)].drop_duplicates(subset=["seq_allele"]).copy()

    # recognition outputs
    seq_overlap=df_col_set_intersection(dataset_train, df_matched_allele_length_recog, "seq", print_output=True, return_output=True)
    df_matched_allele_length_recog=df_matched_allele_length_recog[~df_matched_allele_length_recog.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_val, df_matched_allele_length_recog, "seq", print_output=True, return_output=True)
    df_matched_allele_length_recog=df_matched_allele_length_recog[~df_matched_allele_length_recog.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_test, df_matched_allele_length_recog, "seq", print_output=True, return_output=True)
    df_matched_allele_length_recog=df_matched_allele_length_recog[~df_matched_allele_length_recog.seq.isin(seq_overlap)]

    # activation outputs
    seq_overlap=df_col_set_intersection(dataset_train, df_matched_allele_length_activation, "seq", print_output=True, return_output=True)
    df_matched_allele_length_activation=df_matched_allele_length_activation[~df_matched_allele_length_activation.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_val, df_matched_allele_length_activation, "seq", print_output=True, return_output=True)
    df_matched_allele_length_activation=df_matched_allele_length_activation[~df_matched_allele_length_activation.seq.isin(seq_overlap)]
    seq_overlap=df_col_set_intersection(dataset_test, df_matched_allele_length_activation, "seq", print_output=True, return_output=True)
    df_matched_allele_length_activation=df_matched_allele_length_activation[~df_matched_allele_length_activation.seq.isin(seq_overlap)]

    print(dataset, "recognition and activation dfs")
    print_data_stats(df_matched_allele_length_recog)
    print_data_stats(df_matched_allele_length_activation)
    print(end="\n")

    dataset_cleaned_and_grouped_iedb_data[dataset]={"recognition": df_matched_allele_length_recog, "activation": df_matched_allele_length_activation}

# get recovery of peptides
datasets=["yd", "episcan_r9", "episcan_r9rt", "ba_con", "escape_con"]

# keep separate, easier for plotting
# imm_recovery_over_thresholds_grouped={}
imm_recovery_over_thresholds_recog={}
imm_recovery_over_thresholds_activ={}

for dataset in datasets:

    df_matched_allele_length_recog=dataset_cleaned_and_grouped_iedb_data[dataset]["recognition"].copy()
    df_matched_allele_length_activation=dataset_cleaned_and_grouped_iedb_data[dataset]["activation"].copy()

    n=0 # NOTE: for recovery, use all alleles as we aren't averaging across alleles
    df_matched_allele_length_recog=filter_alleles_min_samples(df_matched_allele_length_recog, n, allele_col="allele")
    df_matched_allele_length_activation=filter_alleles_min_samples(df_matched_allele_length_activation, n, allele_col="allele")

    print("Recog. alleles", df_matched_allele_length_recog.allele.nunique())
    print("Activ. alleles", df_matched_allele_length_activation.allele.nunique())

    dataset_prefix_map={
        "mhcprime": "mhcprime_abt",
        "ft": dataset_prefix_dict[dataset][0],
        "pepcl": dataset_prefix_dict[dataset][1],
    }
    # temp_prefix_map={n:n for n in dataset_prefix_dict[dataset]}
    # dataset_prefix_map = dataset_prefix_map | temp_prefix_map

    print(dataset_prefix_map)

    thresholds=[90, 95, 98, 99, 99.5]
    base_col=None
    # base_col="mhcprime_abt_rank_r_chp" # get delta values
    per_allele=False # do this on aggregate, otherwise small prevalence alleles can interfere

    recog_recovery_df=recovery_table_over_thresholds(df_matched_allele_length_recog, dataset_rank_score_cols[dataset], thresholds, dataset_prefix_map, allele_col="allele", base_col=base_col, per_allele=per_allele)

    activ_recovery_df=recovery_table_over_thresholds(df_matched_allele_length_activation, dataset_rank_score_cols[dataset], thresholds, dataset_prefix_map, allele_col="allele", base_col=base_col, per_allele=per_allele)

    imm_recovery_over_thresholds_recog[dataset]=recog_recovery_df
    imm_recovery_over_thresholds_activ[dataset]=activ_recovery_df

    final_recognition_datasets[dataset]=df_matched_allele_length_recog
    final_activation_datasets[dataset]=df_matched_allele_length_activation

Merging EpiScan R9/RT sets...
Total: 2,498,576
Pos: 108,158 | 4.33%
Neg: 2,390,418 | 95.67%
N alleles: 4
Total: 33,800
Pos: 338 | 1.00%
Neg: 33,462 | 99.00%
N alleles: 4
Total: 80,000
Pos: 800 | 1.00%
Neg: 79,200 | 99.00%
N alleles: 4

yd 19 ['mhcprime_abt', 'holec_ft_steps_300_lr_2eN4_seed_0', 'holec_ft_steps_300_lr_2eN4_seed_1', 'holec_ft_steps_300_lr_2eN4_seed_2', 'holec_ft_steps_300_lr_2eN4_seed_3', 'holec_ft_steps_300_lr_2eN4_seed_4', 'holec_ft_steps_300_lr_2eN4_seed_5', 'holec_ft_steps_300_lr_2eN4_seed_6', 'holec_ft_steps_300_lr_2eN4_seed_7', 'holec_ft_steps_300_lr_2eN4_seed_8', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_0', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_1', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_2', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_3', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_4', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_5', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_6', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_7', 'holec_pepcl_r_dr_steps_300_lr_2eN4_seed_8']
epi

In [ ]:
# add iedb test sets
collapsed_dataset_count_tables,collapsed_dataset_count_summary=(
    append_iedb_tcell_counts(
        count_tables=collapsed_dataset_count_tables,
        summary_df=collapsed_dataset_count_summary,
        recognition_sets=final_recognition_datasets,
        activation_sets=final_activation_datasets,
    )
)

Appended IEDB T cell assay sets:
  YD | T cell Recognition Assays (IEDB): 16 alleles | 2,725 positive | 0 negative | 2,725 total
  EpiScan R9 | T cell Recognition Assays (IEDB): 4 alleles | 1,584 positive | 0 negative | 1,584 total
  EpiScan R9/RT | T cell Recognition Assays (IEDB): 4 alleles | 1,584 positive | 0 negative | 1,584 total
  BA | T cell Recognition Assays (IEDB): 55 alleles | 4,481 positive | 0 negative | 4,481 total
  ESCAPE-seq | T cell Recognition Assays (IEDB): 35 alleles | 3,828 positive | 0 negative | 3,828 total
  YD | T cell Activation Assays (IEDB): 15 alleles | 2,927 positive | 0 negative | 2,927 total
  EpiScan R9 | T cell Activation Assays (IEDB): 4 alleles | 1,207 positive | 0 negative | 1,207 total
  EpiScan R9/RT | T cell Activation Assays (IEDB): 4 alleles | 1,207 positive | 0 negative | 1,207 total
  BA | T cell Activation Assays (IEDB): 77 alleles | 4,759 positive | 0 negative | 4,759 total
  ESCAPE-seq | T cell Activation Assays (IEDB): 41 alleles | 4,40

In [ ]:
# add EpiScan cov, seen and unseen, MS pathogen sets and ligand.MHC atlas sets at the end as well

ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)
ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

sd3_formatted["seq_len"]=sd3_formatted.seq.str.replace("$", "").apply(len)
sd3_formatted_matched_allele_length=sd3_formatted[sd3_formatted.allele.isin(episcan_r9_test_set.allele.unique())].query("seq_len == 9")
sd3_formatted_unseen_allele_matched_length=sd3_formatted[~sd3_formatted.allele.isin(episcan_r9_test_set.allele.unique())].query("seq_len == 9")

ms_test_data["seq_len"]=ms_test_data.seq.str.replace("$", "").apply(len)
ms_test_data_matched_sd3_cov=ms_test_data[ms_test_data.allele.isin(sd3_formatted_matched_allele_length.allele.unique())].query("seq_len == 9")
ms_test_data_unseen_allele_sd3_cov=ms_test_data[ms_test_data.allele.isin(sd3_formatted_unseen_allele_matched_length.allele.unique())].query("seq_len == 9")

# pos and neg counts, should be empty, only do total counts, and no pos to neg ratio, do 0:0
pathogen_set_datasets_dir= DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets" / "Pathogen_Sets"
ms_cov_set=pd.read_csv(f"{pathogen_set_datasets_dir}/ms_cov_set_yd_opt.csv", index_col=0)
ms_zika_set=pd.read_csv(f"{pathogen_set_datasets_dir}/ms_zika_set_yd_opt.csv", index_col=0)
ms_mtb_set=pd.read_csv(f"{pathogen_set_datasets_dir}/ms_mtb_set_yd_opt.csv", index_col=0)

# all pos
escape_seq_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "ESCAPE_seq"
escapeseq_reported_neoantigens_expanded=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded_opt.csv", index_col=0)
escapeseq_reported_neoantigens_expanded=full_process_data(escapeseq_reported_neoantigens_expanded, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
escapeseq_reported_neoantigens_expanded=create_seq_allele_column(escapeseq_reported_neoantigens_expanded)

# ligand mhc atlas set
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
ligand_mhc_atlas_cancer_test_set_for_escape_seq=pd.read_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq_opt.csv", index_col=0)

# get train set for recleaning
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data_remaining.csv", index_col=0)

ms_train_data=full_process_data(ms_train_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data=create_seq_allele_column(ms_train_data)

# reclean properly, missed samples last time from full
seq_overlap=df_col_set_intersection(ms_train_data, ligand_mhc_atlas_cancer_test_set_for_escape_seq, "seq", print_output=True, return_output=True)
ligand_mhc_atlas_cancer_test_set_for_escape_seq=ligand_mhc_atlas_cancer_test_set_for_escape_seq[~ligand_mhc_atlas_cancer_test_set_for_escape_seq.seq.isin(seq_overlap)]
print_data_stats(ligand_mhc_atlas_cancer_test_set_for_escape_seq)

11540
Total: 10,803
Pos: 10,803 | 100.00%
Neg: 0 | 0.00%
N alleles: 41


In [ ]:
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)
ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

sd3_formatted["seq_len"]=sd3_formatted.seq.str.replace("$", "").apply(len)
sd3_formatted_matched_allele_length=sd3_formatted[sd3_formatted.allele.isin(episcan_r9_test_set.allele.unique())].query("seq_len == 9")
sd3_formatted_unseen_allele_matched_length=sd3_formatted[~sd3_formatted.allele.isin(episcan_r9_test_set.allele.unique())].query("seq_len == 9")

ms_test_data["seq_len"]=ms_test_data.seq.str.replace("$", "").apply(len)
ms_test_data_matched_sd3_cov=ms_test_data[ms_test_data.allele.isin(sd3_formatted_matched_allele_length.allele.unique())].query("seq_len == 9")
ms_test_data_unseen_allele_sd3_cov=ms_test_data[ms_test_data.allele.isin(sd3_formatted_unseen_allele_matched_length.allele.unique())].query("seq_len == 9")

In [ ]:
# add special case datasets
collapsed_dataset_count_tables,collapsed_dataset_count_summary=(
    append_special_evaluation_set_counts(
        count_tables=collapsed_dataset_count_tables,
        summary_df=collapsed_dataset_count_summary,

        episcan_cov_matched=sd3_formatted_matched_allele_length,
        episcan_cov_matched_ms=ms_test_data_matched_sd3_cov,
        episcan_cov_transfer=sd3_formatted_unseen_allele_matched_length,
        episcan_cov_transfer_ms=ms_test_data_unseen_allele_sd3_cov,

        ms_cov_ma=ms_cov_set,
        ms_zika_ma=ms_zika_set,
        ms_mtb_ma=ms_mtb_set,

        escape_reported_neoantigens=escapeseq_reported_neoantigens_expanded,
        ligand_mhc_atlas=ligand_mhc_atlas_cancer_test_set_for_escape_seq,
    )
)

Added/updated specialized evaluation sets:
  EpiScan SARS-CoV-2 Matched Allele-Length Test Set: 3 alleles | 129 positive | 29,523 negative | 29,652 total
  EpiScan SARS-CoV-2 Matched Allele-Length Test Set | Matched MS: 3 alleles | 300 positive | 29,700 negative | 30,000 total
  EpiScan SARS-CoV-2 Allele Transfer Learning Test Set: 8 alleles | 262 positive | 78,810 negative | 79,072 total
  EpiScan SARS-CoV-2 Allele Transfer Learning Test Set | Matched MS: 8 alleles | 800 positive | 79,200 negative | 80,000 total
  MS CoV MA Test Set: 9 alleles | 186 total | positive/negative = not reported
  MS Zika MA Test Set: 3 alleles | 57 total | positive/negative = not reported
  MS MTB MA Test Set: 21 alleles | 96 total | positive/negative = not reported
  ESCAPE-seq Reported Neoantigens Test Set: 46 alleles | 225 positive | 0 negative | 225 total
  MS Ligand.MHC Atlas Test Set: 41 alleles | 10,803 positive | 0 negative | 10,803 total


In [ ]:
# rename
collapsed_dataset_count_tables,collapsed_dataset_count_summary=(
    rename_cv_count_table_keys(
        collapsed_dataset_count_tables,
        collapsed_dataset_count_summary,
        verbose=True,
    )
)

YD | CV splits (n=5) | Assay Train
  -> YD | Assay Train | 5 CV
YD | CV splits (n=5) | Matched MS Train
  -> YD | MS Train | 5 CV
YD | CV splits (n=5) | Assay Validation
  -> YD | Assay Val | 5 CV
YD | CV splits (n=5) | Matched MS Validation
  -> YD | MS Val | 5 CV
EpiScan R9 | CV splits (n=5) | Assay Train
  -> EpiScan R9 | Assay Train | 5 CV
EpiScan R9 | CV splits (n=5) | Matched MS Train
  -> EpiScan R9 | MS Train | 5 CV
EpiScan R9/RT | CV splits (n=5) | Assay Train
  -> EpiScan R9/RT | Assay Train | 5 CV
EpiScan R9/RT | CV splits (n=5) | Matched MS Train
  -> EpiScan R9/RT | MS Train | 5 CV
EpiScan R9 | CV splits (n=5) | Assay Validation
  -> EpiScan R9 | Assay Val | 5 CV
EpiScan RT | CV splits (n=5) | Assay Validation
  -> EpiScan RT | Assay Val | 5 CV
EpiScan R9/RT | CV splits (n=5) | Matched MS Validation
  -> EpiScan R9/RT | MS Val | 5 CV
BA | CV splits (n=5) | Assay Train
  -> BA | Assay Train | 5 CV
BA | CV splits (n=5) | Matched MS Train
  -> BA | MS Train | 5 CV
BA | CV spl

In [ ]:
excel_key_replacements={
    # ============================================================
    # CV TRAIN / VALIDATION SETS
    # ============================================================
    "YD | Assay Train | 5 CV":
        "YD Train",
    "YD | MS Train | 5 CV":
        "YD Matched MS Train",
    "YD | Assay Val | 5 CV":
        "YD Val",
    "YD | MS Val | 5 CV":
        "YD Matched MS Val",

    "EpiScan R9 | Assay Train | 5 CV":
        "EpiScan R9 Train",
    "EpiScan R9 | MS Train | 5 CV":
        "EpiScan R9 Matched MS Train",

    "EpiScan R9/RT | Assay Train | 5 CV":
        "EpiScan R9RT Train",
    "EpiScan R9/RT | MS Train | 5 CV":
        "EpiScan R9RT Matched MS Train",

    "EpiScan R9 | Assay Val | 5 CV":
        "EpiScan R9 Val",
    "EpiScan RT | Assay Val | 5 CV":
        "EpiScan RT Val",
    "EpiScan R9/RT | MS Val | 5 CV":
        "EpiScan R9RT Matched MS Val",

    "BA | Assay Train | 5 CV":
        "BA Train",
    "BA | MS Train | 5 CV":
        "BA Matched MS Train",
    "BA | Assay Val | 5 CV":
        "BA Val",
    "BA | MS Val | 5 CV":
        "BA Matched MS Val",

    "ESCAPE-seq | Assay Train | 5 CV":
        "ESCAPE-seq Train",
    "ESCAPE-seq | MS Train | 5 CV":
        "ESCAPE-seq Matched MS Train",
    "ESCAPE-seq | Assay Val | 5 CV":
        "ESCAPE-seq Val",
    "ESCAPE-seq | MS Val | 5 CV":
        "ESCAPE-seq Matched MS Val",

    # ============================================================
    # FIXED TEST SETS
    # ============================================================
    "YD | Test | Assay":
        "YD Test",
    "YD | Test | Matched MS":
        "YD Test Matched MS",

    "EpiScan R9 | Test | Assay":
        "EpiScan R9 Test",
    "EpiScan R9 | Test | Matched MS":
        "EpiScan R9 Test Matched MS",

    "EpiScan RT | Test | Assay":
        "EpiScan RT Test",
    "EpiScan RT | Test | Matched MS":
        "EpiScan RT Test Matched MS",

    "BA | Test | Assay":
        "BA Test",
    "BA | Test | Matched MS":
        "BA Test Matched MS",

    "ESCAPE-seq | Test | Assay":
        "ESCAPE-seq Test",
    "ESCAPE-seq | Test | Matched MS":
        "ESCAPE-seq Test Matched MS",

    # ============================================================
    # MULTI-DATASET UPDATING
    # ============================================================
    "YD Multi-dataset Updating | Train":
        "YD MD Train",
    "EpiScan R9/RT Multi-dataset Updating | Train":
        "EpiScan R9RT MD Train",
    "BA Multi-dataset Updating | Train":
        "BA MD Train",
    "ESCAPE-seq Multi-dataset Updating | Train":
        "ESCAPE-seq MD Train",

    "YD Multi-dataset Updating | Test":
        "YD MD Test",
    "EpiScan R9/RT Multi-dataset Updating | Test":
        "EpiScan R9RT MD Test",
    "BA Multi-dataset Updating | Test":
        "BA MD Test",
    "ESCAPE-seq Multi-dataset Updating | Test":
        "ESCAPE-seq MD Test",

    # ============================================================
    # IEDB T CELL SETS
    # ============================================================
    "YD | T cell Recognition Assays (IEDB)":
        "YD T cell recog.",
    "EpiScan R9 | T cell Recognition Assays (IEDB)":
        "EpiScan R9 T cell recog.",
    "EpiScan R9/RT | T cell Recognition Assays (IEDB)":
        "EpiScan R9RT T cell recog.",
    "BA | T cell Recognition Assays (IEDB)":
        "BA T cell recog.",
    "ESCAPE-seq | T cell Recognition Assays (IEDB)":
        "ESCAPE-seq T cell recog.",

    "YD | T cell Activation Assays (IEDB)":
        "YD T cell act.",
    "EpiScan R9 | T cell Activation Assays (IEDB)":
        "EpiScan R9 T cell act.",
    "EpiScan R9/RT | T cell Activation Assays (IEDB)":
        "EpiScan R9RT T cell act.",
    "BA | T cell Activation Assays (IEDB)":
        "BA T cell act.",
    "ESCAPE-seq | T cell Activation Assays (IEDB)":
        "ESCAPE-seq T cell act.",

    # ============================================================
    # SARS-COV-2 / TRANSFER SETS
    # ============================================================
    "EpiScan SARS-CoV-2 Matched Allele-Length Test Set":
        "EpiScan CoV-2 Matched Test",

    "EpiScan SARS-CoV-2 Matched Allele-Length Test Set | Matched MS":
        "EpiScan CoV-2 Matched MS",

    "EpiScan SARS-CoV-2 Allele Transfer Learning Test Set":
        "EpiScan CoV-2 Transfer Test",

    "EpiScan SARS-CoV-2 Allele Transfer Learning Test Set | Matched MS":
        "EpiScan CoV-2 Transfer MS",

    # ============================================================
    # OTHER SPECIAL TEST SETS
    # ============================================================
    "ESCAPE-seq Reported Neoantigens Test Set":
        "ESCAPE-seq Reported Neoantigens",
}

collapsed_dataset_count_tables,collapsed_dataset_count_summary=(
    replace_count_table_keys(
        count_tables=collapsed_dataset_count_tables,
        summary_df=collapsed_dataset_count_summary,
        replacement_dict=excel_key_replacements,
        verbose=True,
    )
)

YD | Assay Train | 5 CV
  -> YD Train
YD | MS Train | 5 CV
  -> YD Matched MS Train
YD | Assay Val | 5 CV
  -> YD Val
YD | MS Val | 5 CV
  -> YD Matched MS Val
YD | Test | Assay
  -> YD Test
YD | Test | Matched MS
  -> YD Test Matched MS
EpiScan R9 | Assay Train | 5 CV
  -> EpiScan R9 Train
EpiScan R9 | MS Train | 5 CV
  -> EpiScan R9 Matched MS Train
EpiScan R9/RT | Assay Train | 5 CV
  -> EpiScan R9RT Train
EpiScan R9/RT | MS Train | 5 CV
  -> EpiScan R9RT Matched MS Train
EpiScan R9 | Assay Val | 5 CV
  -> EpiScan R9 Val
EpiScan RT | Assay Val | 5 CV
  -> EpiScan RT Val
EpiScan R9/RT | MS Val | 5 CV
  -> EpiScan R9RT Matched MS Val
EpiScan R9 | Test | Assay
  -> EpiScan R9 Test
EpiScan R9 | Test | Matched MS
  -> EpiScan R9 Test Matched MS
EpiScan RT | Test | Assay
  -> EpiScan RT Test
EpiScan RT | Test | Matched MS
  -> EpiScan RT Test Matched MS
BA | Assay Train | 5 CV
  -> BA Train
BA | MS Train | 5 CV
  -> BA Matched MS Train
BA | Assay Val | 5 CV
  -> BA Val
BA | MS Val | 5 CV


In [ ]:
# save as multi-sheet excel
from pathlib import Path
import pandas as pd

# output directory
supplementary_data_dir=f"{FIGURES_DIR}/Supplementary_Data"

output_dir=Path(supplementary_data_dir)
output_dir.mkdir(parents=True,exist_ok=True)

output_path=output_dir/"train_validation_test_dataset_split_counts.xlsx"

def make_safe_excel_sheet_name(name,used_names):
    """Sanitize, truncate, and ensure unique Excel sheet names."""
    safe=(
        str(name)
        .replace("/","-")
        .replace("\\","-")
        .replace("*","")
        .replace("?","")
        .replace(":","-")
        .replace("[","(")
        .replace("]",")")
    )

    safe=safe[:31]
    base=safe
    i=2

    while safe in used_names:
        suffix=f"_{i}"
        safe=f"{base[:31-len(suffix)]}{suffix}"
        i+=1

    used_names.add(safe)
    return safe


used_names=set()

with pd.ExcelWriter(output_path,engine="openpyxl") as writer:

    # ============================================================
    # FIRST SHEET: SUMMARY
    # ============================================================
    summary_sheet=make_safe_excel_sheet_name(
        "Summary",
        used_names,
    )

    collapsed_dataset_count_summary.to_excel(
        writer,
        sheet_name=summary_sheet,
        index=False,
    )

    # ============================================================
    # ALL PER-ALLELE COUNT TABLES
    # ============================================================
    for dataset_name,df in collapsed_dataset_count_tables.items():

        safe_name=make_safe_excel_sheet_name(
            dataset_name,
            used_names,
        )

        df.to_excel(
            writer,
            sheet_name=safe_name,
            index=False,
        )

print(f"Saved: {output_path}")

Saved: /home/pchati/PepCL_Manuscript/Figures/Supplementary_Data/train_validation_test_dataset_split_counts.xlsx
